# 🎲 02 · Probability: the language of uncertainty
### Carriers, genotypes and traits as random variables; joint and conditional probability; Bayes' rule for genetic tests; (conditional) independence; expectation and correlation; Hardy–Weinberg; the Gaussian; counting parameters

**BiologyToML** · Level 1 · 02-620 core · Part A — Foundations: from biology to data · Module 02 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/02_probability_for_biologists.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy (`scipy.stats`, only to check our formulas), pandas, Matplotlib |
| **Prerequisites** | Modules 00–01 (running notebooks; Python, NumPy, pandas, Matplotlib). No probability or statistics course needed. |
| **Estimated time** | 150 min (reading + running + the ✏️ exercises) |
| **02-620 topics covered** | The probability prerequisite of 02-620 (CMU 02-680) and the probability the lectures review: Bernoulli and multinoulli distributions (naïve Bayes); joint, marginal and conditional probability, the chain rule; Bayes' rule in machine learning — prior, likelihood, posterior (introductory lecture); independence and conditional independence, the smoking network (Bayesian networks); why full joint tables need exponentially many parameters and how conditional independence rescues us (naïve Bayes, Bayesian networks) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> Everything here is **simulated** so that we know the true probabilities and can check every rule
>     numerically. Test characteristics (sensitivity, specificity, prevalence), the smoking/yellow-nails/lung-cancer
>     probabilities and the genotype–disease risks are **illustrative numbers**, not clinical estimates. The gene–gene
>     correlations of Section 5 use the toolkit's **SIMULATED** leukaemia data (shaped like Golub *et al.* 1999). The only
>     published figures are the approximate cystic-fibrosis incidence of Section 6 and a classic census table of
>     wealth, gender and working hours typed in for an exercise (Section 2.3).

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Describe** biological quantities as random variables and **simulate** Bernoulli, categorical, binomial and Gaussian distributions, distinguishing a probability mass function from a density.
2. **Compute** joint, marginal and conditional probabilities from a table, and **apply** the sum, product and chain rules.
3. **Derive** Bayes' rule and **use** it to compute the probability of disease after a positive test and the carrier risk of a relative; **name** prior, likelihood, posterior and evidence.
4. **Distinguish** independence from conditional independence, and **demonstrate** with the smoking → yellow nails / lung cancer network of the Bayesian-network lecture that two variables can be dependent yet independent given a third.
5. **Compute** expectations, variances, covariances and correlations (true and sample versions), and **explain** with a simulated population-structure confounder why correlation is not causation.
6. **Derive** the Hardy–Weinberg proportions from the product and sum rules and **check** them against data.
7. **Use** the Gaussian (68–95–99.7 rule, z-scores, central limit theorem, 2-D correlation) to describe polygenic traits and co-expressed genes.
8. **Count** the parameters of a joint distribution (2ⁿ − 1, 3ⁿ − 1) and **explain** how conditional independence makes models such as naïve Bayes tractable.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports and course helpers | 5 min |
| 1 · Random variables and distributions | carrier status, genotypes, allele counts, a quantitative trait | 25 min |
| 2 · Joint, marginal, conditional | a genotype × disease table; sum, product and chain rules | 18 min |
| 3 · Bayes' rule | a screening test, the base-rate fallacy, a cystic-fibrosis family | 22 min |
| 4 · Independence and conditional independence | smoking, yellow nails and lung cancer; naïve Bayes | 20 min |
| 5 · Expectation, variance, correlation | genotype moments, gene–gene correlations (🧪 simulated truth), a confounder | 20 min |
| 6 · Hardy–Weinberg as a probability model | p², 2pq, q² from first principles; carriers of cystic fibrosis | 10 min |
| 7 · The Gaussian in detail | 68–95–99.7, z-scores, the central limit theorem, 2-D Gaussians | 17 min |
| 8 · Counting parameters | why full joint tables explode and structure rescues us | 8 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 5 min |

## 🧫 The biological question

A couple is expecting a child. The woman's brother has **cystic fibrosis**, a recessive disease caused by mutations in
the *CFTR* gene; she herself is healthy. What is the probability that she carries a disease allele? Her partner has
no family history — what is *his* risk? And if a prenatal screening test comes back positive, how worried should they
be? Genetic counsellors answer such questions every day, and the answers are often surprising: a positive result
from a test that is "99 % accurate" can still mean the disease is unlikely.

The tool behind these answers is **probability**, the mathematics of uncertainty. It is also the language of
02-620: the course formulates supervised learning as learning $P(Y \mid X)$ and unsupervised learning as learning
$P(X)$ (Module 00), uses **Bayes' rule** to turn data into beliefs about parameters, and builds naïve Bayes, Bayesian
networks and hidden Markov models on **conditional independence**. Even genetics itself is probabilistic: Mendel's laws are statements about chance, and the bell-shaped
distribution of height arises because many genes each add a small random contribution.

This module builds that language from zero, with genotypes, carriers, diagnostic tests and gene expression. Every rule
is checked by simulation: if a formula is right, a computer experiment with many random individuals must agree with it.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
from math import comb, factorial

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
`simulate_genotypes` draws genotypes under Hardy–Weinberg equilibrium (Sections 6–8),
               `simulate_leukemia_expression` generates the leukaemia-like expression data for Section 5,
               `draw_graph` draws small networks and `CLASS_COLORS` keeps the course's colours. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import os
import numpy as np
import matplotlib.pyplot as plt

LEUKEMIA_MARKERS = {
    "CST3": ("AML", "cystatin C"), "ZYX": ("AML", "zyxin"), "CD33": ("AML", "CD33 antigen"),
    "MPO": ("AML", "myeloperoxidase"), "LYZ": ("AML", "lysozyme"), "CTSD": ("AML", "cathepsin D"),
    "FAH": ("AML", "fumarylacetoacetate hydrolase"), "LTC4S": ("AML", "leukotriene C4 synthase"),
    "LYN": ("AML", "LYN kinase"), "HOXA9": ("AML", "homeobox A9"), "CFD": ("AML", "adipsin"),
    "LEPR": ("AML", "leptin receptor"), "SRGN": ("AML", "proteoglycan 1 (serglycin)"),
    "CXCL8": ("AML", "interleukin-8"), "AZU1": ("AML", "azurocidin"), "MCL1": ("AML", "MCL1"),
    "ITGAX": ("AML", "CD11c"), "NFKBIA": ("AML", "MAD-3 (IκBα)"), "ELANE": ("AML", "neutrophil elastase"),
    "CEBPA": ("AML", "C/EBPα"), "CSF3R": ("AML", "G-CSF receptor"), "FCER1G": ("AML", "Fc receptor γ"),
    "TYROBP": ("AML", "DAP12"), "S100A9": ("AML", "calprotectin subunit"), "CFP": ("AML", "properdin"),
    "MYB": ("ALL", "c-Myb"), "CCND3": ("ALL", "cyclin D3"), "RBBP4": ("ALL", "RbAp48"),
    "SMARCA4": ("ALL", "SNF2 (BRG1)"), "TCF3": ("ALL", "E2A"), "DUT": ("ALL", "dUTPase"),
    "STMN1": ("ALL", "Op18 (stathmin)"), "IL7R": ("ALL", "IL-7 receptor"), "RABEP1": ("ALL", "rabaptin-5"),
    "CBX3": ("ALL", "heterochromatin protein p25"), "DNTT": ("ALL", "terminal deoxynucleotidyl transferase"),
    "CD24": ("ALL", "CD24"), "TOP2B": ("ALL", "topoisomerase II β"), "IRF2": ("ALL", "IRF-2"),
    "MYL6": ("ALL", "myosin light chain"), "PSMB8": ("ALL", "proteasome subunit"), "LMO2": ("ALL", "LMO2"),
    "CD79A": ("B-ALL", "MB-1 (Igα)"), "CD19": ("B-ALL", "CD19"), "CD79B": ("B-ALL", "Igβ"),
    "VPREB1": ("B-ALL", "pre-B receptor"), "PAX5": ("B-ALL", "PAX5"), "CD22": ("B-ALL", "CD22"),
    "CD3E": ("T-ALL", "CD3ε"), "CD3D": ("T-ALL", "CD3δ"), "CD2": ("T-ALL", "CD2"), "CD7": ("T-ALL", "CD7"),
    "LCK": ("T-ALL", "LCK kinase"), "CD1E": ("T-ALL", "CD1e"),
}


def simulate_leukemia_expression(seed: int = 1999) -> dict:
    """Synthetic leukaemia expression data shaped like the classic Golub et al. (1999) study.

    72 bone-marrow/blood samples — 47 acute lymphoblastic leukaemia (ALL: 38 B-lineage, 9 T-lineage)
    and 25 acute myeloid leukaemia (AML) — × 3,571 genes on a log10 microarray scale, with the
    original 38/34 training/test design (train: 19 B-ALL, 8 T-ALL, 11 AML).  Values below the
    detection limit are floored at 2.0 (≈ 23 % of entries) and saturate at 4.2.  The 54 genes of
    ``LEUKEMIA_MARKERS`` carry the strongest signals (42 differ between ALL and AML by 0.5–1.0
    log10 units; 12 mark B- or T-lineage ALL), 900 more genes differ modestly between ALL and AML,
    150 separate T-ALL from B-ALL, and 15 hidden co-expression programmes correlate genes.  Two
    AML samples are atypical (signature at 35 % strength).  The classes are cleaner than in the
    real study, so many classifiers make few or no errors.

    Returns a dict with ``X`` (72 × 3,571), ``y`` (0 = ALL, 1 = AML), ``labels``, ``subtype``,
    ``genes``, ``is_train``, ``sample_id`` (1–72), ``atypical`` (True for the two weak-signature AML
    samples), ``true_effect`` and ``source``.  ``true_effect`` is the planted AML − ALL difference
    per gene; it is 0 for the 12 lineage markers and the 150 T-vs-B genes, although the B-lineage
    markers still differ between ALL and AML through the subtype shifts (by about 0.8 log10 units).
    The default ``seed`` gives the course's standard data set.
    """
    rng = np.random.default_rng(seed)
    J = 3571
    subtype = np.array(["B-ALL"] * 19 + ["T-ALL"] * 8 + ["AML"] * 11 + ["B-ALL"] * 19 + ["T-ALL"] * 1 + ["AML"] * 14,
                       dtype=object)
    is_train = np.r_[np.ones(38, bool), np.zeros(34, bool)]
    order = np.r_[rng.permutation(38), 38 + rng.permutation(34)]          # mix classes within each set
    subtype, is_train = subtype[order], is_train[order]
    n = len(subtype)
    y = (subtype == "AML").astype(int)
    named = list(LEUKEMIA_MARKERS)
    genes = np.array(named + [f"G{j:04d}" for j in range(1, J - len(named) + 1)], dtype=object)
    low = rng.random(J) < 0.4                                             # genes mostly below detection
    mu = np.where(low, rng.uniform(1.5, 2.4, J), np.clip(rng.normal(2.9, 0.4, J), 2.2, 4.0))
    sd = rng.uniform(0.12, 0.35, J)
    X = mu + sd * rng.normal(0, 1, (n, J))
    factors = rng.normal(0, 1, (n, 15))                                  # co-expression programmes
    loadings = rng.normal(0, 0.15, (15, J)) * (rng.random((15, J)) < 0.08)
    X += factors @ loadings
    effect = np.zeros(J)
    strong = np.arange(len(named))
    for j, g in enumerate(named):
        cls = LEUKEMIA_MARKERS[g][0]
        size = rng.uniform(0.5, 1.0)
        effect[j] = size if cls == "AML" else -size if cls == "ALL" else 0.0
        mu[j] = max(mu[j], 2.5)
    moderate = rng.choice(np.arange(len(named), J), 900, replace=False)
    effect[moderate] = rng.choice([-1, 1], 900) * rng.uniform(0.1, 0.4, 900)
    strength = np.ones(n)
    strength[np.flatnonzero(y == 1)[rng.choice((y == 1).sum(), 2, replace=False)]] = 0.35   # atypical AML
    X[:, strong] = mu[strong] + sd[strong] * rng.normal(0, 1, (n, len(strong)))
    X += np.outer((y - y.mean()) * strength, effect)
    for g, (cls, _) in LEUKEMIA_MARKERS.items():
        if cls in ("B-ALL", "T-ALL"):
            j = named.index(g)
            X[:, j] += np.where(subtype == cls, 0.9, np.where(subtype == "AML", -0.2, -0.5))
    lineage = rng.choice(np.setdiff1d(np.arange(len(named), J), moderate), 150, replace=False)
    X[:, lineage] += np.outer(np.where(subtype == "T-ALL", 1.0, np.where(subtype == "B-ALL", -0.2, 0.0)),
                              rng.choice([-1, 1], 150) * rng.uniform(0.4, 0.8, 150))
    X = np.round(np.clip(X, 2.0, np.log10(16000)), 4)
    return dict(X=X, y=y, labels=np.where(y == 1, "AML", "ALL").astype(object), subtype=subtype, genes=genes,
                is_train=is_train, sample_id=np.arange(1, n + 1), atypical=strength < 1, true_effect=effect,
                source="SIMULATED leukaemia data (shaped like Golub et al. 1999)")


def simulate_genotypes(rng: np.random.Generator, n_samples: int, n_snps: int, n_pops: int = 1,
                       fst: float = 0.0, maf_range=(0.05, 0.5)):
    """Simulate 0/1/2 genotypes under Hardy–Weinberg equilibrium.

    Each SNP gets an ancestral allele frequency drawn uniformly from ``maf_range``.  With
    ``n_pops > 1`` and ``fst > 0`` every population's frequency is drawn from the
    Balding–Nichols model Beta(p(1−F)/F, (1−p)(1−F)/F).  Samples are split evenly across
    populations.  Returns ``(G, pop, freqs)``: the N × J genotype matrix, the population of
    each sample and the n_pops × J matrix of population allele frequencies.

    ``G`` is stored as int8 to save memory: convert with ``G.astype(float)`` before arithmetic
    whose intermediate results can exceed 127.  A rare SNP can come out monomorphic (no
    variation) in a small sample — skip such columns in per-SNP tests.
    """
    p = rng.uniform(*maf_range, n_snps)
    if n_pops > 1 and fst > 0:
        a, b = p * (1 - fst) / fst, (1 - p) * (1 - fst) / fst
        freqs = np.clip(rng.beta(a, b, size=(n_pops, n_snps)), 0.001, 0.999)
    else:
        freqs = np.tile(p, (n_pops, 1))
    pop = np.repeat(np.arange(n_pops), int(np.ceil(n_samples / n_pops)))[:n_samples]
    G = rng.binomial(2, freqs[pop]).astype(np.int8)
    return G, pop, freqs


CLASS_COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e", "#8c564b"]


def draw_graph(edges, pos: dict, directed: bool = True, ax=None, highlight=None, node_color: str = "#9ecae1",
               highlight_color: str = "#fdae6b", radius: float = 0.07, title: str = "", edge_labels=None):
    """Draw a small graph with Matplotlib only.

    ``edges`` is a list of (u, v) pairs, ``pos`` maps every node to (x, y) in [0, 1]².
    ``highlight`` is a set of nodes to colour differently (e.g. observed evidence).
    """
    from matplotlib.patches import Circle, FancyArrowPatch
    ax = ax or plt.gca()
    highlight = set(highlight or [])
    for i, (u, v) in enumerate(edges):
        (x0, y0), (x1, y1) = pos[u], pos[v]
        d = np.hypot(x1 - x0, y1 - y0) or 1.0
        sx, sy = x0 + (x1 - x0) * radius / d, y0 + (y1 - y0) * radius / d
        ex, ey = x1 - (x1 - x0) * radius / d, y1 - (y1 - y0) * radius / d
        ax.add_patch(FancyArrowPatch((sx, sy), (ex, ey), arrowstyle="-|>" if directed else "-",
                                     mutation_scale=12, lw=1.2, color="0.25"))
        if edge_labels is not None:
            ax.text((x0 + x1) / 2, (y0 + y1) / 2, str(edge_labels[i]), fontsize=7, color="0.2",
                    ha="center", va="center", bbox=dict(fc="white", ec="none", alpha=0.7, pad=0.5))
    for node, (x, y) in pos.items():
        ax.add_patch(Circle((x, y), radius, fc=highlight_color if node in highlight else node_color,
                            ec="0.2", lw=1, zorder=3))
        ax.text(x, y, str(node), ha="center", va="center", fontsize=9, zorder=4)
    ax.set_xlim(-0.1, 1.1); ax.set_ylim(-0.1, 1.1); ax.set_aspect("equal"); ax.axis("off")
    if title:
        ax.set_title(title)
    return ax


print("🧰 Toolkit loaded:", ", ".join(['simulate_leukemia_expression', 'simulate_genotypes', 'draw_graph']))

---
## 1 · Random variables and distributions

> 📍 **Where we are.** In Module 01 you simulated coin flips and genotypes with `rng`. Now we name what you were doing:
> each simulated quantity was a **random variable**, and the rule that generated it was a **distribution**.

A **random variable** is a quantity whose value is uncertain before we observe it: whether the next person we test is a
carrier, the genotype of a newborn at a SNP, the expression level of a gene in a new patient. We write random variables
with capital letters ($X$, $G$, $Y$) and particular values with lower-case letters ($x$, $g$, $y$).

A **probability** is a number between 0 (impossible) and 1 (certain). $P(X = x)$ is read "the probability that $X$ takes
the value $x$". Two basic requirements: every probability is between 0 and 1, and the probabilities of all possible
values **add up to 1** — something must happen.

A random variable is **discrete** if it takes separate values (0/1, AA/AT/TT, a count) and **continuous** if it can
take any value in a range (height, log-expression). The list of probabilities of a discrete variable is its
**probability mass function (pmf)**.

### 1.1 Bernoulli: carrier or not

The simplest random variable has two values, 1 ("yes") and 0 ("no"), with $P(X = 1) = \theta$. This is the
**Bernoulli distribution**. In the notation of the naïve-Bayes lecture (Module 08, where $\pi$ plays the role of
$\theta$):

$$P(X = x) = \theta^{x}\,(1-\theta)^{1-x}, \qquad x \in \{0, 1\}.$$

*In words:* if $x = 1$ the formula gives $\theta^1 (1-\theta)^0 = \theta$; if $x = 0$ it gives $\theta^0(1-\theta)^1 = 1 - \theta$ (anything to the power 0 is 1). One formula covers both cases — a trick 02-620 uses for every discrete
distribution. Example: whether a randomly chosen person of Northern-European ancestry carries a cystic-fibrosis
allele, with $\theta \approx 0.04$ (roughly 1 in 25; Section 6 derives this number).

In [ ]:
# ── 1.1 Bernoulli: simulate carrier status ──
theta = 0.04                                         # P(carrier) ≈ 1 in 25
rng = np.random.default_rng(SEED + 1)
for n in [25, 1000, 100_000]:
    carriers = rng.random(n) < theta                 # True with probability theta
    print(f"{n:7d} people: {carriers.sum():5d} carriers, fraction {carriers.mean():.4f}  (true {theta})")
print("pmf from the formula: P(X=1) =", theta ** 1 * (1 - theta) ** 0, " P(X=0) =", theta ** 0 * (1 - theta) ** 1)

🔍 **What just happened?** In a group of 25 people the number of carriers is very variable (0, 1, 2 or 3 are all
plausible); with 100,000 people the observed fraction is within a few thousandths of 0.04. The observed fraction is our
**estimate** of $\theta$, and it becomes reliable as the sample grows — the **law of large numbers**, which we will see
again and again.

### 1.2 Categorical: a genotype with three values

At a SNP with alleles A and T, a person's genotype takes three values; we code them as in the regression lecture:
AA = 0, AT = 1, TT = 2 (the number of T alleles). A variable with $K$ categories and probabilities
$\theta_0, \dots, \theta_{K-1}$ that sum to 1 is **categorical** (02-620 calls it **multinoulli**). The Bernoulli trick
extends to it with an **indicator function** $\mathbb{I}[x = k]$, which is 1 if $x = k$ and 0 otherwise:

$$P(X = x) = \prod_{k=0}^{K-1} \theta_k^{\,\mathbb{I}[x = k]}.$$

*In words:* all factors but one have exponent 0 and equal 1; the surviving factor is $\theta_x$, the probability of the
value that occurred. Naïve Bayes (Module 08) writes the probability of a genotype exactly this way. Below we use the
genotype probabilities $(0.49, 0.42, 0.09)$, which correspond to a T-allele frequency of 0.3 — Section 6 explains why.

In [ ]:
# ── 1.2 Categorical: simulate genotypes AA/AT/TT ──
genotype_probs = np.array([0.49, 0.42, 0.09])        # P(G = 0), P(G = 1), P(G = 2); they sum to 1
print("probabilities sum to", round(genotype_probs.sum(), 12))      # round away tiny floating-point fuzz
rng = np.random.default_rng(SEED + 2)
G_sample = rng.choice([0, 1, 2], size=1000, p=genotype_probs)
observed = np.bincount(G_sample, minlength=3) / len(G_sample)         # fraction of 0s, 1s, 2s
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(np.arange(3) - 0.2, genotype_probs, width=0.4, color="tab:blue", label="true pmf")
ax.bar(np.arange(3) + 0.2, observed, width=0.4, color="tab:orange", label="1,000 simulated people")
ax.set_xticks([0, 1, 2]); ax.set_xticklabels(["AA (0)", "AT (1)", "TT (2)"])
ax.set_ylabel("probability / fraction"); ax.set_title("categorical distribution of a genotype"); ax.legend()
plt.tight_layout(); plt.show()
print("observed fractions:", observed.round(3))

🔍 **What just happened?** `rng.choice` drew 1,000 genotypes with the given probabilities, and `np.bincount` counted
each value. The orange bars (observed fractions) are close to the blue bars (the pmf) but not identical — sampling
noise again. A bar chart is the natural picture of a pmf: one bar per value, heights summing to 1.

### 1.3 Binomial: counting successes

How many T alleles does a person carry? Each of the two chromosomes independently carries T with probability $q$
(the **allele frequency**), so the genotype is the number of "successes" in 2 trials. More generally, the number of
successes $K$ in $n$ independent trials, each with success probability $\theta$, follows the **binomial distribution**:

$$P(K = k) = \binom{n}{k}\,\theta^{k}\,(1-\theta)^{n-k}, \qquad k = 0, 1, \dots, n.$$

🧮 **Math refresher — factorials and "n choose k".** $n!$ ("n factorial") is $n \times (n-1) \times \dots \times 2 \times 1$, with $0! = 1$; e.g. $4! = 24$. The **binomial coefficient** $\binom{n}{k} = \frac{n!}{k!\,(n-k)!}$ ("n choose k")
counts the ways of choosing which $k$ of the $n$ trials are the successes. For $n = 2$: $\binom{2}{0} = 1$,
$\binom{2}{1} = 2$, $\binom{2}{2} = 1$.

*In words:* any particular sequence with $k$ successes and $n-k$ failures has probability $\theta^k(1-\theta)^{n-k}$
(multiply the probabilities of the individual trials — Section 2 explains why multiplying is allowed for independent
events), and there are $\binom{n}{k}$ such sequences. For a diploid genotype with $q = 0.3$:
$P(G=1) = 2 \times 0.3 \times 0.7 = 0.42$ — exactly the middle value of Section 1.2.

A second example: in a clinic that sees $n = 100$ unrelated patients, how many are CF carriers? $K \sim \text{Binomial}(100, 0.04)$ (the symbol $\sim$ reads "is distributed as").

In [ ]:
# ── 1.3 Binomial pmf by hand, checked against SciPy and against simulation ──
def binomial_pmf(k: int, n: int, theta: float) -> float:
    """P(K = k) for K ~ Binomial(n, theta), straight from the formula."""
    return comb(n, k) * theta ** k * (1 - theta) ** (n - k)


print("factorial(4) =", factorial(4), "| comb(2, 1) =", comb(2, 1), "| comb(100, 4) =", f"{comb(100, 4):,}")
print("diploid genotype, q = 0.3:", [round(binomial_pmf(k, 2, 0.3), 3) for k in range(3)])

n_patients, theta = 100, 0.04
ks = np.arange(0, 13)
ours = np.array([binomial_pmf(k, n_patients, theta) for k in ks])
print("largest difference from scipy.stats.binom.pmf:", np.abs(ours - stats.binom.pmf(ks, n_patients, theta)).max())

rng = np.random.default_rng(SEED + 3)
clinics = rng.binomial(n_patients, theta, size=10_000)        # 10,000 clinics, 100 patients each
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(ks, np.bincount(clinics, minlength=13)[:13] / len(clinics), color="tab:orange", alpha=0.7, label="10,000 simulated clinics")
ax.plot(ks, ours, "o-", color="tab:blue", label="binomial pmf (formula)")
ax.set_xlabel("number of carriers among 100 patients"); ax.set_ylabel("probability")
ax.set_title("Binomial(100, 0.04)"); ax.legend()
plt.tight_layout(); plt.show()
print(f"P(no carrier at all) = {binomial_pmf(0, 100, 0.04):.3f};  P(at least 8 carriers) = {1 - ours[:8].sum():.3f}")

🔍 **What just happened?** Our formula agrees with SciPy to about $10^{-16}$ (floating-point rounding), and the
simulated clinics follow the pmf closely. The most likely count is 3 or 4 carriers among 100 patients, but 0 carriers
(probability ≈ 0.017) and 8 or more (≈ 0.05) both happen. A clinic that sees 8 carriers has not necessarily discovered
anything unusual — **chance alone produces surprising-looking counts**, a lesson that matters every time we compare
groups.

### 1.4 Continuous variables and the Gaussian density

Height, blood pressure and log-expression are **continuous**. For a continuous variable the probability of any *exact*
value is zero (nobody is exactly 170.000000… cm tall); instead we describe it by a **probability density function
(pdf)** $p(x)$, and probabilities are **areas** under the density: $P(a < X < b)$ is the area under $p(x)$ between $a$
and $b$. The most important density is the **Gaussian** (or **normal**) distribution with mean $\mu$ and standard
deviation $\sigma$:

$$p(x) = \frac{1}{\sigma\sqrt{2\pi}}\,\exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right).$$

🧮 **Math refresher — exp, π and areas.** $\exp(z) = e^{z}$, where $e \approx 2.718$; $e^0 = 1$ and $e^{z}$ shrinks
towards 0 as $z$ becomes very negative. $\pi \approx 3.1416$. The "area under a curve" between $a$ and $b$ is written
$\int_a^b p(x)\,dx$ (an **integral**); you can approximate it by cutting the region into thin rectangles of width
$\Delta x$ and adding their areas $p(x)\Delta x$. That is all we need here.

*In words:* the density is highest at $x = \mu$ and falls off symmetrically; $(x-\mu)^2/\sigma^2$ measures how many
standard deviations $x$ is from the centre, squared; the factor $1/(\sigma\sqrt{2\pi})$ makes the total area exactly 1.

⚠️ **A density is not a probability.** A density can be larger than 1. Log10-expression values with $\sigma = 0.1$
have a peak density of $1/(0.1\sqrt{2\pi}) \approx 4$ — yet the total area is still 1, because the curve is narrow.

In [ ]:
# ── 1.4 The Gaussian density: formula, areas, and a density above 1 ──
def gaussian_pdf(x, mu: float, sigma: float):
    """Gaussian density N(mu, sigma²) evaluated at x (works on arrays)."""
    return np.exp(-(x - mu) ** 2 / (2 * sigma ** 2)) / (sigma * np.sqrt(2 * np.pi))


mu_h, sd_h = 170.0, 7.0                                   # an illustrative adult height distribution (cm)
dx = 0.01
x = np.arange(100, 240, dx)
area_total = np.sum(gaussian_pdf(x, mu_h, sd_h) * dx)     # thin rectangles
inside = (x > 163) & (x < 177)
area_band = np.sum(gaussian_pdf(x[inside], mu_h, sd_h) * dx)
print(f"total area under the curve: {area_total:.4f}")
print(f"P(163 < height < 177) by rectangles: {area_band:.4f} | by scipy.stats.norm.cdf: "
      f"{stats.norm.cdf(177, mu_h, sd_h) - stats.norm.cdf(163, mu_h, sd_h):.4f}")
print(f"peak density with sigma = 7 cm: {gaussian_pdf(mu_h, mu_h, sd_h):.4f}; with sigma = 0.1: {gaussian_pdf(0, 0, 0.1):.3f}")

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(x, gaussian_pdf(x, mu_h, sd_h), color="tab:blue")
ax.fill_between(x[inside], gaussian_pdf(x[inside], mu_h, sd_h), color="tab:blue", alpha=0.3, label="P(163 < X < 177) = area")
ax.set_xlim(135, 205); ax.set_xlabel("height x (cm)"); ax.set_ylabel("density p(x) (per cm)")
ax.set_title("Gaussian density N(170, 7²): probability = area"); ax.legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** Adding up thin rectangles gives a total area of 1.0000 and $P(163 < X < 177) \approx 0.683$,
matching SciPy's exact value. That interval is $\mu \pm \sigma$: about **68 %** of a Gaussian lies within one standard
deviation of the mean (Section 7 completes the "68–95–99.7 rule"). The density's units are "probability per cm"; with
$\sigma = 0.1$ the peak is 3.989, above 1, as promised.

### 1.5 Histograms approach the distribution

If we simulate many values and draw a **histogram** scaled so that the bar areas add to 1 (`density=True`), the
histogram approaches the pdf as the sample grows — the continuous version of the law of large numbers.

In [ ]:
# ── 1.5 Histograms of simulated heights approach the Gaussian density ──
rng = np.random.default_rng(SEED + 5)
fig, axes = plt.subplots(1, 3, figsize=(11, 3), sharey=True)
for ax, n in zip(axes, [20, 200, 20_000]):
    sample = rng.normal(mu_h, sd_h, n)
    ax.hist(sample, bins=30, range=(140, 200), density=True, color="lightgray", edgecolor="gray")
    ax.plot(x, gaussian_pdf(x, mu_h, sd_h), color="tab:red")
    ax.set_xlim(140, 200); ax.set_xlabel("height (cm)")
    ax.set_title(f"n = {n:,}: mean {sample.mean():.1f}, sd {sample.std():.1f}")
axes[0].set_ylabel("density")
plt.tight_layout(); plt.show()

🔍 **What just happened?** With 20 people the histogram is ragged and the sample mean (in the title) is off by more
than a centimetre; with 20,000 the histogram hugs the red curve and the sample statistics are
within about 0.1 cm of $\mu = 170$ and $\sigma = 7$. The **sample mean** and **sample standard deviation** are estimates of the true
parameters — Section 5 defines them; Module 05 asks which estimates are *best*.

✏️ **Your turn.** (1) In cell 1.1 set `theta = 0.5`: how variable is the fraction of carriers among 25 people now?
(2) In cell 1.3 compute the probability that a clinic of 100 sees **no** carrier when $\theta = 0.01$. (3) In cell 1.4,
compute $P(\text{height} > 185)$ with rectangles and with `stats.norm.sf(185, mu_h, sd_h)` (`sf` = 1 − cdf).

---
## 2 · Joint, marginal and conditional probability

> 📍 **Where we are.** Section 1 treated one random variable at a time. Biology is about *relationships*: does a
> genotype change disease risk? Here we describe two (or three) random variables together.

Consider a SNP genotype $G \in \{0, 1, 2\}$ and a disease status $D \in \{0, 1\}$ (0 = healthy, 1 = affected). We
simulate a cohort of 5,000 people in which the T allele raises risk — the risks are illustrative.

* The **joint probability** $P(G = g, D = d)$ — "G is g **and** D is d" — is the fraction of the population in that
  cell of the table.
* The **marginal probability** $P(G = g)$ ignores $D$: add up the joint probabilities over all values of $D$. This is
  the **sum rule** (also called marginalisation):
  $$P(G = g) = \sum_{d} P(G = g, D = d).$$
  *In words:* the probability of a genotype is the probability of having it *and* being healthy plus the probability of
  having it *and* being affected. (The name "marginal" comes from writing these sums in the margin of the table.)
* The **conditional probability** $P(D = d \mid G = g)$ — "the probability of D = d **given** G = g" — restricts
  attention to the people with genotype $g$:
  $$P(D = d \mid G = g) = \frac{P(G = g, D = d)}{P(G = g)}.$$
  *In words:* among the people with genotype $g$, what fraction have disease status $d$? The vertical bar "|" reads
  "given".

In [ ]:
# ── 2.1 Simulate a cohort and build the genotype × disease table ──
rng = np.random.default_rng(SEED + 21)
N_cohort = 5000
G_c = rng.choice([0, 1, 2], size=N_cohort, p=[0.49, 0.42, 0.09])        # genotypes (T-allele frequency 0.3)
risk = np.array([0.05, 0.08, 0.15])                                       # ILLUSTRATIVE P(D = 1 | G = g)
D_c = (rng.random(N_cohort) < risk[G_c]).astype(int)                      # each person's disease status
counts = pd.crosstab(pd.Series(G_c, name="genotype G"), pd.Series(D_c, name="disease D"))
counts.index = ["AA (0)", "AT (1)", "TT (2)"]; counts.columns = ["healthy (0)", "affected (1)"]
print("counts (SIMULATED cohort):"); print(counts)
joint = counts / N_cohort                                                  # joint probabilities
print("\njoint probabilities P(G, D):"); print(joint.round(4))
print("sum of all joint probabilities:", round(joint.to_numpy().sum(), 12))

🔍 **What just happened?** `pd.crosstab` counted how many people fall in each of the 3 × 2 cells. Dividing by 5,000
turned counts into joint probabilities, which add up to 1. (Without the `round`, Python would print
0.9999999999999999: computers store decimals with tiny rounding errors of about $10^{-16}$ — read such numbers as
exactly 1.) The table is dominated by healthy AA and AT people;
affected TT people are rare because TT itself is rare. Now the marginals and conditionals.

In [ ]:
# ── 2.2 Marginals (sum rule) and conditionals ──
P_G = joint.sum(axis=1)                                   # sum over D → P(G)
P_D = joint.sum(axis=0)                                   # sum over G → P(D)
print("P(G) =", P_G.round(4).to_dict())
print("P(D) =", P_D.round(4).to_dict())
P_D_given_G = joint.div(P_G, axis=0)                      # each row divided by its row total
print("\nP(D | G)  (each ROW sums to 1):"); print(P_D_given_G.round(4))
P_G_given_D = joint.div(P_D, axis=1)                      # each column divided by its column total
print("\nP(G | D)  (each COLUMN sums to 1):"); print(P_G_given_D.round(4))

🔍 **What just happened?** The marginal $P(G)$ is close to the genotype probabilities we simulated (0.49, 0.42, 0.09),
and 7 % of the cohort is affected. The conditional table $P(D \mid G)$ estimates the risks we built in (5 %, 8 %,
15 %): we get 4.6 %, 8.7 % and 11.5 %. The TT estimate is furthest off because only 469 people are TT — an estimate
based on few people is noisy. Risk clearly rises with the number of T alleles.

⚠️ **$P(D \mid G)$ is not $P(G \mid D)$.** $P(D = 1 \mid G = \text{AA}) \approx 0.05$: very few AA people are affected.
But $P(G = \text{AA} \mid D = 1) \approx 0.32$: almost a third of the *affected* people are AA, simply because AA is so
common. Confusing the two is one of the commonest errors in reading medical and genetic studies. A case–control GWAS
samples people by $D$ and therefore measures $P(G \mid D)$; Bayes' rule (Section 3) converts between them.

### 2.3 The product rule and the chain rule

Rearranging the definition of conditional probability gives the **product rule**:

$$P(G, D) = P(D \mid G)\,P(G) = P(G \mid D)\,P(D).$$

*In words:* the probability of "G and D" is the probability of G times the probability of D once G is known — or the
other way round; both orders are correct. Applying it repeatedly to three variables gives the **chain rule** (the
starting point of both the naïve-Bayes and the Bayesian-network lectures):

$$P(A, B, C) = P(A)\,P(B \mid A)\,P(C \mid A, B).$$

*In words:* draw A first, then B given A, then C given both. The chain rule holds for **any** distribution and any
order of the variables; it makes no assumption. Let us check both rules numerically, adding a third variable — smoking
$S$ — to our cohort.

In [ ]:
# ── 2.3 Check the product rule and the chain rule on the simulated cohort ──
lhs = joint.loc["TT (2)", "affected (1)"]
via_G = P_D_given_G.loc["TT (2)", "affected (1)"] * P_G["TT (2)"]
via_D = P_G_given_D.loc["TT (2)", "affected (1)"] * P_D["affected (1)"]
print(f"P(G=TT, D=1) = {lhs:.4f} | P(D=1|G=TT)·P(G=TT) = {via_G:.4f} | P(G=TT|D=1)·P(D=1) = {via_D:.4f}")

S_c = (rng.random(N_cohort) < np.where(D_c == 1, 0.45, 0.25)).astype(int)   # smokers: more common among affected (ILLUSTRATIVE)
df3 = pd.DataFrame({"G": G_c, "D": D_c, "S": S_c})
p_joint3 = ((df3.G == 2) & (df3.D == 1) & (df3.S == 1)).mean()               # P(G=2, D=1, S=1) directly
p_G = (df3.G == 2).mean()
p_D_given_G = ((df3.G == 2) & (df3.D == 1)).mean() / p_G
p_S_given_GD = ((df3.G == 2) & (df3.D == 1) & (df3.S == 1)).mean() / ((df3.G == 2) & (df3.D == 1)).mean()
print(f"P(G=2, D=1, S=1) = {p_joint3:.5f} | chain rule P(G)·P(D|G)·P(S|G,D) = {p_G * p_D_given_G * p_S_given_GD:.5f}")

🔍 **What just happened?** All three ways of computing $P(G=\text{TT}, D=1)$ agree, and the chain-rule product for
three variables reproduces the directly counted joint probability. With frequencies computed from the same data these
identities hold *exactly* — they are rules of arithmetic on proportions, not approximations. Their power comes later:
models in Modules 08 and 15–20 replace some of the conditional factors by simpler ones, and *that* is where
assumptions enter (Section 4).

✏️ **Your turn.** The introductory lecture of 02-620 illustrates "learn the joint distribution, then condition on the
inputs" with a classic census table $P(W, G, H)$ of wealth (poor/rich), gender and hours worked per week: female ≤ 40.5 h: poor 0.253122, rich 0.0245895; female > 40.5 h: poor 0.0421768, rich 0.0116293; male ≤ 40.5 h:
poor 0.331313, rich 0.0971295; male > 40.5 h: poor 0.134106, rich 0.105933. Put the eight numbers in a NumPy array of
shape (2, 2, 2) and compute (a) the marginal $P(W = \text{rich})$ and (b) $P(W = \text{rich} \mid G = \text{male}, H > 40.5)$. (Answers in the ✅ Checkpoint; Module 00 computed $P(W = \text{rich} \mid G = \text{female}, H > 40.5) = 0.216$ from
the same table.) This is the lecture's recipe for supervised learning through a joint distribution: *learn the joint,
then condition*.

---
## 3 · Bayes' rule

> 📍 **Where we are.** Section 2 warned that $P(D \mid G) \neq P(G \mid D)$. Bayes' rule is the bridge between them —
> and the engine of the Bayesian half of 02-620 (MAP estimation in Modules 05 and 07, naïve Bayes in Module 08,
> MCMC in Module 21).

Write the product rule both ways and divide:

$$P(A \mid B) = \frac{P(B \mid A)\,P(A)}{P(B)}, \qquad P(B) = \sum_{a} P(B \mid A = a)\,P(A = a).$$

*In words:* to reverse a conditional probability, multiply by the probability of what you want to learn ($P(A)$) and
divide by the overall probability of what you observed ($P(B)$, computed with the sum and product rules).

In the introductory lecture of 02-620 the same rule is written for learning: $D$ = the observed data (the **evidence**) and $\theta$ =
an unknown quantity (a model parameter, a diagnosis, a prediction):

$$\underbrace{P(\theta \mid D)}_{\text{posterior}} = \frac{\overbrace{P(D \mid \theta)}^{\text{likelihood}}\;\overbrace{P(\theta)}^{\text{prior}}}{\underbrace{P(D)}_{\text{evidence}}}.$$

*In words:* the **prior** is your belief about $\theta$ *before* seeing the data; the **likelihood** says how probable
the observed data are if $\theta$ were true; the **posterior** is your updated belief *after* seeing the data; the
**evidence** $P(D)$ is the same for every $\theta$ and just makes the posterior sum to 1.

### 3.1 A screening test

A disease affects 1 % of the population (the **prevalence**). A test detects it in 99 % of affected people (the
**sensitivity**, $P(+ \mid \text{disease})$) and correctly gives a negative result in 95 % of healthy people (the
**specificity**, $P(- \mid \text{healthy})$). These numbers are illustrative but typical of screening. A person tests
positive. What is the probability they have the disease? *Before you run the next cell, write down your guess.*

In [ ]:
# ── 3.1 P(disease | positive test) by Bayes' rule ──
def posterior_positive(prevalence: float, sensitivity: float, specificity: float) -> float:
    """P(disease | +) = P(+ | disease) P(disease) / P(+), with P(+) from the sum rule."""
    p_pos = sensitivity * prevalence + (1 - specificity) * (1 - prevalence)     # evidence P(+)
    return sensitivity * prevalence / p_pos


prev, sens, spec = 0.01, 0.99, 0.95
p_pos = sens * prev + (1 - spec) * (1 - prev)
print(f"P(+) = {sens}·{prev} + {1 - spec:.2f}·{1 - prev} = {p_pos:.4f}")
print(f"P(disease | +) = {sens}·{prev} / {p_pos:.4f} = {posterior_positive(prev, sens, spec):.4f}")
print(f"P(disease | −) = {(1 - sens) * prev / (1 - p_pos):.6f}")

🔍 **What just happened?** Despite "99 % sensitivity", a positive result means only a **1 in 6 (16.7 %)** chance of
disease. A negative result, in contrast, is very reassuring (about 1 in 10,000). Most people guess something near 95 %;
the gap is called the **base-rate fallacy** — forgetting that the disease is rare (the prior). Among 100 people, 1 is
sick; among the 99 healthy ones, 5 % ≈ 5 test positive anyway. So positives are mostly false alarms. The next cell
shows this with simulated people ("natural frequencies").

In [ ]:
# ── 3.2 Simulate 100,000 people and count ──
rng = np.random.default_rng(SEED + 32)
n_people = 100_000
sick = rng.random(n_people) < prev
positive = np.where(sick, rng.random(n_people) < sens, rng.random(n_people) < 1 - spec)
table = pd.crosstab(pd.Series(np.where(sick, "sick", "healthy"), name="truth"),
                    pd.Series(np.where(positive, "test +", "test −"), name="result"))
print(table)
print(f"\nfraction of positives who are sick: {sick[positive].mean():.4f}  (Bayes: {posterior_positive(prev, sens, spec):.4f})")

🔍 **What just happened?** About 1,000 of the 100,000 simulated people are sick and nearly all of them test positive;
but about 5,000 healthy people *also* test positive. The fraction of positives who are truly sick (0.161) is close to
the Bayes value (0.167); the small gap is sampling noise.
Counting people instead of juggling probabilities is the easiest way to explain a test result to a patient.

### 3.3 The prior matters — and posteriors become priors

In [ ]:
# ── 3.3 P(disease | +) against prevalence, and a second independent test ──
prevalences = np.logspace(-4, -0.3, 200)                  # from 0.0001 to 0.5
fig, ax = plt.subplots(figsize=(6.5, 3.4))
for sp, colour in [(0.95, "tab:blue"), (0.99, "tab:orange"), (0.999, "tab:green")]:
    ax.plot(prevalences, posterior_positive(prevalences, 0.99, sp), color=colour, label=f"specificity {sp}")
ax.axvline(0.01, color="0.5", ls=":", lw=1)
ax.set_xscale("log"); ax.set_xlabel("prevalence = prior P(disease)  (log scale)"); ax.set_ylabel("P(disease | +)")
ax.set_title("the same test means very different things in different populations (sensitivity 0.99)")
ax.legend(); plt.tight_layout(); plt.show()

after_first = posterior_positive(prev, sens, spec)
after_second = posterior_positive(after_first, sens, spec)   # the posterior after test 1 is the prior for test 2
print(f"after one positive test: {after_first:.3f};  after a second, independent positive test: {after_second:.3f}")

🔍 **What just happened?** For a rare disease (left side of the plot) even a very good test produces mostly false
positives; in a high-risk group (right side) the same result is nearly conclusive. Improving specificity from 0.95 to
0.999 matters enormously when the prior is small. And Bayes' rule can be applied **repeatedly**: the posterior after
one positive test (0.167) becomes the prior for a second test, which raises the probability to about **0.80**.
⚠️ That last step assumes the two test results are *independent given the disease status* — if both tests are fooled
by the same thing (say, a related harmless variant), the second positive adds much less. Conditional independence is
the topic of Section 4.

### 3.4 Bayes in a family: is the healthy sister a carrier?

Back to the couple of the introduction. Cystic fibrosis is **recessive**: write N for the normal allele and c for a
disease allele; only cc people are affected. The woman's brother is affected (cc), so both parents must be carriers
(Nc). Each child of Nc × Nc parents is NN, Nc or cc with probabilities 1/4, 1/2, 1/4 (Mendel). The woman is healthy —
that is the **evidence**: she is not cc. Bayes' rule with $\theta$ = her genotype:

$$P(\text{Nc} \mid \text{healthy}) = \frac{P(\text{healthy} \mid \text{Nc})\,P(\text{Nc})}{P(\text{healthy})}
= \frac{1 \times \tfrac12}{\tfrac14 + \tfrac12} = \frac{2}{3}.$$

*In words:* of the three equally likely healthy "slots" (NN, Nc, cN), two are carriers. Her prior risk as a child of
two carriers was 1/2; learning that she is healthy *raised* it to 2/3, because it rules out cc but not the carrier
states.

In [ ]:
# ── 3.4 Simulate 100,000 children of two carrier parents ──
rng = np.random.default_rng(SEED + 34)
n_children = 100_000
from_mother = rng.random(n_children) < 0.5                 # True = she passes on the disease allele c
from_father = rng.random(n_children) < 0.5
n_c = from_mother.astype(int) + from_father.astype(int)    # 0 = NN, 1 = Nc, 2 = cc
healthy = n_c < 2
print(f"fraction affected (cc): {np.mean(n_c == 2):.4f}  (Mendel: 0.25)")
print(f"P(carrier | healthy) by simulation: {np.mean(n_c[healthy] == 1):.4f}  (Bayes: {2 / 3:.4f})")

p_partner = 0.04                                           # population carrier frequency (Section 6)
p_child = (2 / 3) * p_partner * (1 / 4)                    # both carriers, AND the child gets c from both
print(f"P(their child is affected) = 2/3 · {p_partner} · 1/4 = {p_child:.4f}  ≈ 1 in {1 / p_child:.0f}")

🔍 **What just happened?** The simulation confirms the 1/4 Mendelian ratio and the 2/3 posterior. Combining it with
her partner's population carrier risk (≈ 0.04, derived in Section 6) and the 1/4 chance that two carriers have an
affected child gives a risk of about **1 in 150** for the couple's child — the kind of number a genetic counsellor
writes down. (A negative carrier test for the partner would lower it further: Bayes again, with the test's sensitivity.)

✏️ **Your turn.** (1) In cell 3.1, a confirmatory test has sensitivity 0.95 and specificity 0.999. Starting from the
posterior 0.167 after the screening test, what is the probability of disease after a positive confirmatory test?
(2) The partner takes a carrier test that detects 90 % of carrier alleles (sensitivity 0.9, specificity 1.0) and is
negative. Use Bayes' rule to update his carrier probability from 0.04. What is the couple's new risk?

### 📘 Lecture notes — Bayes' rule in machine learning

**Bayes' rule** follows from writing the product rule both ways, $P(\theta, D) = P(\theta \mid D)P(D) = P(D \mid \theta)P(\theta)$:

$$P(\theta \mid D) = \frac{P(D \mid \theta)\,P(\theta)}{P(D)}, \qquad P(D) = \sum_{\theta'} P(D \mid \theta')\,P(\theta').$$

*In words:* posterior = likelihood × prior ÷ evidence. **D** is the data (the evidence); **θ** the unknown
quantity — a model parameter, a diagnosis, a prediction. The **prior** $P(\theta)$ is the belief before seeing D;
the **likelihood** $P(D \mid \theta)$ says how probable the observed data are under θ; the **posterior**
$P(\theta \mid D)$ is the belief after seeing D; the **evidence** $P(D)$ is the same for every θ and makes the
posterior sum to 1.

**Two ways to learn parameters** (the introductory lecture; Module 05 in depth):

- **MLE** (frequentist): $\hat\theta_{\text{MLE}} = \arg\max_\theta P(D \mid \theta)$ — uses the data only; called
  "objective".
- **MAP** (Bayesian): $\hat\theta_{\text{MAP}} = \arg\max_\theta P(D \mid \theta)\,P(\theta)$ — combines data and
  belief; criticised as "subjective". The evidence does not depend on θ, so it drops out of the maximisation.

*Worked examples (this section):* screening test with prevalence 0.01, sensitivity 0.99, specificity 0.95 →
$P(+) = 0.99 \cdot 0.01 + 0.05 \cdot 0.99 = 0.0594$ and $P(\text{disease} \mid +) = 0.0099 / 0.0594 = 0.167$; a
second independent positive test → 0.80; the healthy sister of a cystic-fibrosis patient →
$P(\text{carrier} \mid \text{healthy}) = \tfrac{1 \cdot 1/2}{1/4 + 1/2} = 2/3$.

**Common confusions.** $P(D \mid \theta) \neq P(\theta \mid D)$; the likelihood is not a probability distribution
over θ (it need not sum to 1 over θ); a posterior can become the prior for the next observation only if the
observations are conditionally independent given θ.

> 🎓 **Exam tip.** Always identify which quantity is the prior, which the likelihood, and compute P(D) by summing likelihood ×
    prior over *all* values of θ. A frequent exam error is to report the likelihood P(+ | disease) when the question
    asks for the posterior P(disease | +).

---
## 4 · Independence and conditional independence

> 📍 **Where we are.** The two-test calculation of Section 3.3 needed an assumption: the tests are independent *given*
> the disease status. This section makes that idea precise. It is the single most important idea for the
> probabilistic half of 02-620: naïve Bayes (Module 08), Bayesian networks (Modules 15–17) and hidden Markov models
> (Modules 19–20) are all built from conditional-independence assumptions.

**Independence.** Two random variables A and B are independent if knowing one tells you nothing
about the other:

$$P(A, B) = P(A)\,P(B) \quad\Longleftrightarrow\quad P(A \mid B) = P(A).$$

*In words:* the joint probability factorises into the product of the marginals; equivalently, conditioning on B does not
change the probability of A. (Why are the two forms equivalent? If $P(A \mid B) = P(A)$, then the product rule gives
$P(A, B) = P(A \mid B)P(B) = P(A)P(B)$.) We write $A \perp B$.

**Conditional independence.** A and B are independent **given** C if, once C is
known, B carries no further information about A:

$$P(A, B \mid C) = P(A \mid C)\,P(B \mid C) \quad\Longleftrightarrow\quad P(A \mid B, C) = P(A \mid C).$$

*In words:* within each group of C, A and B behave independently. We write $A \perp B \mid C$. ⚠️ The two notions are
**different**: variables can be dependent but conditionally independent (the example below), or independent but
conditionally dependent (Module 16's "explaining away").

### 4.1 Independent SNPs, dependent genotype and disease

Two SNPs on different chromosomes are inherited independently (Mendel's second law), so their genotypes are
independent. Genotype and disease in our cohort of Section 2 are *not*.

In [ ]:
# ── 4.1 Check P(A, B) = P(A) P(B) for two unlinked SNPs and for genotype × disease ──
rng = np.random.default_rng(SEED + 41)
snp1 = rng.choice(3, size=20_000, p=[0.49, 0.42, 0.09])
snp2 = rng.choice(3, size=20_000, p=[0.64, 0.32, 0.04])     # a second SNP, T-allele frequency 0.2
joint12 = pd.crosstab(snp1, snp2, normalize=True).to_numpy()           # P(SNP1, SNP2)
product12 = np.outer(joint12.sum(axis=1), joint12.sum(axis=0))          # P(SNP1) · P(SNP2) for every cell
print("unlinked SNPs: largest |P(a,b) − P(a)P(b)| =", f"{np.abs(joint12 - product12).max():.4f}")
productGD = np.outer(P_G, P_D)
print("genotype × disease: largest |P(g,d) − P(g)P(d)| =", f"{np.abs(joint.to_numpy() - productGD).max():.4f}")
print("P(D=1) =", round(P_D["affected (1)"], 4), "but P(D=1 | G=TT) =", round(P_D_given_G.loc["TT (2)", "affected (1)"], 4))

🔍 **What just happened?** For the unlinked SNPs the joint table equals the product of the marginals up to sampling
noise (differences of a few thousandths). For genotype and disease the discrepancy is larger, and the conditional
check is clearer: $P(D = 1 \mid G = \text{TT})$ is about 1.6 times the overall $P(D = 1)$. Knowing the genotype changes the
disease probability, so $G$ and $D$ are dependent. ⚠️ With finite samples, "equal" always means "equal up to sampling
noise"; deciding whether a small difference is real is the job of statistical tests (and of Modules 06–07, 18).

### 4.2 Smoking, yellow fingernails and lung cancer

The Bayesian-network lecture of 02-620 opens with three variables: **S**moking, **Y**ellow fingernails and **L**ung
cancer. Smoking causes both yellow fingernails (tar stains) and lung cancer; in the graph there is no arrow between Y
and L, only a dashed line marking that they are *correlated*. Graphs like this — probabilistic graphical models — are
how causal inference describes the distribution of causes and effects (Judea Pearl received the 2011 Turing Award for
this work). The lecture writes the joint distribution as

$$P(S, Y, L) = P(Y \mid S)\,P(L \mid S)\,P(S) \quad\Rightarrow\quad Y \perp L \mid S.$$

*In words:* the chain rule would give $P(S)P(Y \mid S)P(L \mid S, Y)$; the graph says that, once we know whether
someone smokes, their fingernails tell us nothing more about lung cancer, so $P(L \mid S, Y) = P(L \mid S)$. Yet
**without** knowing smoking status, yellow fingernails *do* predict lung cancer — because both point to smoking. The
lecture gives no numbers, so we choose **illustrative** ones (drawn on the graph by the next cell): $P(S{=}1) = 0.2$; $P(Y{=}1 \mid S{=}1) = 0.5$,
$P(Y{=}1 \mid S{=}0) = 0.02$; $P(L{=}1 \mid S{=}1) = 0.15$, $P(L{=}1 \mid S{=}0) = 0.01$.

In [ ]:
# ── 4.2 The smoking network, and exact probabilities from the factorisation P(S) P(Y|S) P(L|S) ──
P_S1 = 0.2
P_Y1_given_S = {1: 0.5, 0: 0.02}                    # ILLUSTRATIVE conditional probability tables
P_L1_given_S = {1: 0.15, 0: 0.01}

fig, ax = plt.subplots(figsize=(4.8, 3.2))
pos = {"S": (0.5, 0.85), "Y": (0.15, 0.2), "L": (0.85, 0.2)}
draw_graph([("S", "Y"), ("S", "L")], pos, ax=ax, radius=0.09)
ax.plot([0.26, 0.74], [0.2, 0.2], ls="--", color="0.5", lw=1)          # dashed line: correlated, but no arrow
ax.text(0.5, 0.25, "correlated", ha="center", fontsize=7, color="0.4")
ax.text(0.62, 0.92, f"P(S=1) = {P_S1}", fontsize=7.5)
ax.text(-0.08, 0.0, f"P(Y=1|S=1) = {P_Y1_given_S[1]}\nP(Y=1|S=0) = {P_Y1_given_S[0]}", fontsize=7.5, va="top")
ax.text(0.65, 0.0, f"P(L=1|S=1) = {P_L1_given_S[1]}\nP(L=1|S=0) = {P_L1_given_S[0]}", fontsize=7.5, va="top")
ax.set_ylim(-0.25, 1.05)
ax.set_title("S = smoking, Y = yellow fingernails, L = lung cancer", fontsize=9)
plt.tight_layout(); plt.show()


def p_joint_SYL(s: int, y: int, l: int) -> float:
    """P(S=s, Y=y, L=l) = P(S) P(Y|S) P(L|S): the factorisation of the smoking network."""
    ps = P_S1 if s else 1 - P_S1
    py = P_Y1_given_S[s] if y else 1 - P_Y1_given_S[s]
    pl = P_L1_given_S[s] if l else 1 - P_L1_given_S[s]
    return ps * py * pl


total = sum(p_joint_SYL(s, y, l) for s in (0, 1) for y in (0, 1) for l in (0, 1))
print("the 8 joint probabilities sum to", round(total, 12))
for y in (1, 0):        # P(L=1 | Y=y): sum over the unobserved S (sum rule), divide by P(Y=y)
    num = sum(p_joint_SYL(s, y, 1) for s in (0, 1))
    den = sum(p_joint_SYL(s, y, l) for s in (0, 1) for l in (0, 1))
    print(f"P(L=1 | Y={y}) = {num:.5f} / {den:.5f} = {num / den:.4f}")
for s in (1, 0):
    for y in (1, 0):
        p = p_joint_SYL(s, y, 1) / (p_joint_SYL(s, y, 0) + p_joint_SYL(s, y, 1))
        print(f"  P(L=1 | S={s}, Y={y}) = {p:.4f}")

🔍 **What just happened?** From three small tables we computed everything. People with yellow fingernails have a
**13 %** chance of lung cancer versus **2.6 %** for people without — a five-fold difference, so **Y and L are
dependent**. But within smokers, $P(L = 1 \mid S = 1, Y)$ is 0.15 whether or not the nails are yellow, and within
non-smokers it is 0.01 either way: **given S, Y adds nothing**. Fingernails "predict" cancer only because they are a
marker of smoking. Now the same with simulated people — the way you would see it in real data.

In [ ]:
# ── 4.3 Simulate 200,000 people from the network and look at the data ──
rng = np.random.default_rng(SEED + 43)
n = 200_000
S = (rng.random(n) < P_S1).astype(int)
Y = (rng.random(n) < np.where(S == 1, P_Y1_given_S[1], P_Y1_given_S[0])).astype(int)   # depends on S only
L = (rng.random(n) < np.where(S == 1, P_L1_given_S[1], P_L1_given_S[0])).astype(int)   # depends on S only

rates = {"all people": [L[Y == 0].mean(), L[Y == 1].mean()],
         "non-smokers (S=0)": [L[(S == 0) & (Y == 0)].mean(), L[(S == 0) & (Y == 1)].mean()],
         "smokers (S=1)": [L[(S == 1) & (Y == 0)].mean(), L[(S == 1) & (Y == 1)].mean()]}
print(pd.DataFrame(rates, index=["P(L=1 | Y=0)", "P(L=1 | Y=1)"]).round(4))
fig, ax = plt.subplots(figsize=(6.5, 3.2))
for k, (group, (r0, r1)) in enumerate(rates.items()):
    ax.bar(k - 0.18, r0, width=0.36, color="tab:blue", label="normal nails (Y=0)" if k == 0 else None)
    ax.bar(k + 0.18, r1, width=0.36, color="gold", edgecolor="k", label="yellow nails (Y=1)" if k == 0 else None)
ax.set_xticks(range(3)); ax.set_xticklabels(list(rates)); ax.set_ylabel("fraction with lung cancer")
ax.set_title("Y and L are dependent, but independent given S (SIMULATED)"); ax.legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** In the whole simulated population (left pair of bars) yellow nails go with a five-fold
higher cancer rate — the dependence a naïve data analyst would discover. Splitting by smoking status (**stratifying**,
or **conditioning** on S) makes the difference vanish up to sampling noise: in each stratum the two bars have the same
height. This is conditional independence in data. It is also the simplest example of **confounding**: S is a common
cause of Y and L. Painting someone's nails yellow would not give them cancer — the dependence is not causal (Section
5.4 returns to this with genetics).

### 4.4 From conditional independence to naïve Bayes

The naïve-Bayes lecture reviews exactly the step that naïve Bayes needs. For two features $X_1, X_2$ and a class $Y$:

$$P(X_1, X_2 \mid Y) \;\overset{\text{chain rule}}{=}\; P(X_1 \mid X_2, Y)\,P(X_2 \mid Y)
\;\overset{X_1 \perp X_2 \mid Y}{=}\; P(X_1 \mid Y)\,P(X_2 \mid Y).$$

*In words:* the chain rule is always true; the second equality is an **assumption** — given the class, the features are
independent. In our network the "class" is S and the features are Y and L. Let us check the factorisation numerically:

In [ ]:
# ── 4.4 Check P(Y, L | S) = P(Y | S) P(L | S) in the simulated data ──
for s in (0, 1):
    in_s = S == s
    joint_YL = pd.crosstab(Y[in_s], L[in_s], normalize=True).to_numpy()          # P(Y, L | S = s)
    product_YL = np.outer(joint_YL.sum(axis=1), joint_YL.sum(axis=0))              # P(Y | S = s) P(L | S = s)
    print(f"S = {s}: largest |P(y,l|s) − P(y|s)P(l|s)| = {np.abs(joint_YL - product_YL).max():.5f}")
joint_all = pd.crosstab(Y, L, normalize=True).to_numpy()
product_all = np.outer(joint_all.sum(axis=1), joint_all.sum(axis=0))
print(f"without conditioning on S: largest |P(y,l) − P(y)P(l)| = {np.abs(joint_all - product_all).max():.5f}")

🔍 **What just happened?** Within each smoking stratum the joint table of (Y, L) equals the product of its marginals to
about $10^{-3}$ or better (sampling noise), while across the whole population the factorisation fails by a larger
margin. That is exactly the factorisation naïve Bayes assumes for genotypes given cancer status in Module 08, where it
shrinks the number of parameters from astronomical to manageable (Section 8).

✏️ **Your turn.** (1) Change `P_Y1_given_S` to `{1: 0.5, 0: 0.5}` (nails no longer depend on smoking) and re-run cells
4.2–4.4. Are Y and L still dependent? (2) A **chain** also produces conditional independence: SNP → gene expression →
disease. Simulate it (expression depends only on the SNP, disease only on expression) and check that SNP and disease
are dependent but independent given expression. (3) Module 16 shows the third possible structure, a **collider**
(A → C ← B), where independent A and B become *dependent* once you condition on C.

### 📘 Lecture notes — independence, conditional independence and a first graphical model

**How to check independence.** A and B are independent ($A \perp B$) if and only if

$$P(A, B) = P(A)\,P(B) \quad\Longleftrightarrow\quad P(A \mid B) = P(A).$$

*In words:* the joint factorises into the marginals; equivalently, learning B does not change the probability of A.
The two forms are equivalent because, if $P(A \mid B) = P(A)$, the product rule gives
$P(A, B) = P(A \mid B)P(B) = P(A)P(B)$.

**How to check conditional independence.** A and B are independent given C ($A \perp B \mid C$) if and only if

$$P(A, B \mid C) = P(A \mid C)\,P(B \mid C) \quad\Longleftrightarrow\quad P(A \mid B, C) = P(A \mid C).$$

*In words:* once C is known, B tells us nothing more about A.

**The smoking network.** S → Y and S → L (a **fork**: one common cause), with a dashed line between Y and L because
they are correlated:

$$P(S, Y, L) = P(S)\,P(Y \mid S)\,P(L \mid S) \;\Rightarrow\; Y \perp L \mid S, \text{ but } Y \not\perp L.$$

*In words:* the joint distribution is built from three small tables — how common smoking is, and how smoking
changes the chances of yellow nails and of lung cancer. Because L's table depends on S only, once smoking status is
known the nails add nothing about cancer; but if smoking status is unknown, yellow nails make smoking — and hence
cancer — more likely.

*Worked example (illustrative numbers of Section 4.2):* $P(L = 1 \mid Y = 1) = 0.13$ vs $P(L = 1 \mid Y = 0) =
0.026$ — dependent; but $P(L = 1 \mid S = 1, Y) = 0.15$ and $P(L = 1 \mid S = 0, Y) = 0.01$ whatever Y is —
conditionally independent. Graphs whose missing edges encode conditional independences are **probabilistic
graphical models**; directed ones are Bayesian networks (Modules 15–17), undirected ones Markov random fields
(Module 18). Naïve Bayes is the simplest Bayesian network.

> 🎓 **Exam tip.** Know both forms of each definition and why they are equivalent. Do not confuse "Y and L are independent"
    (false here) with "Y and L are independent given S" (true here) — exam questions on d-separation (Module 16) test
    exactly this distinction.

### 📘 Lecture notes — the naïve-Bayes factorisation

**The step naïve Bayes needs.** For features $X_1, X_2$ and class $Y$:

$$P(X_1, X_2 \mid Y) \overset{\text{chain rule}}{=} P(X_1 \mid X_2, Y)\,P(X_2 \mid Y) \overset{X_1 \perp X_2 \mid Y}{=} P(X_1 \mid Y)\,P(X_2 \mid Y).$$

*In words:* the first equality is always true; the second is an **assumption** — given the class, the features
carry no information about each other. For $n$ features, $P(X_1, \dots, X_n \mid Y) = \prod_{i=1}^{n} P(X_i \mid Y)$.

**Why it matters — the genotype → cancer example of the lecture.** Predicting cancer status Y (binary) from
genotypes $X_1, \dots, X_n \in \{0, 1, 2\}$: a full table $P(Y \mid X_1, \dots, X_n)$ needs a probability for each of
the $3^n$ genotype combinations ($3^{30} \approx 2 \times 10^{14}$ for 30 SNPs). Bayes' rule alone,
$P(Y \mid X) \propto P(X \mid Y)P(Y)$, does not help: $P(X_1, \dots, X_n \mid Y)$ still needs $(3^n - 1) \times 2$
numbers, plus 1 for $P(Y)$. With the naïve-Bayes assumption each $P(X_i \mid Y = y)$ is a 3-valued (multinoulli)
distribution with 2 free numbers, so the model needs $2 \times n \times 2 + 1$ — for 30 SNPs, **121**.

> 🎓 **Exam tip.** When asked to "derive the naïve Bayes factorisation", write the chain rule first and label explicitly where
    the conditional-independence assumption is used — graders look for that label.

---
## 5 · Expectation, variance, covariance and correlation

> 📍 **Where we are.** A distribution is a whole list of probabilities. Often we summarise it by a few numbers: its
> centre, its spread, and — for two variables — how they move together. These summaries appear in every later module
> (least squares minimises a variance, PCA diagonalises a covariance matrix, Gaussian graphical models invert one).

**Expectation.** The expected value (mean) of a discrete random variable is the probability-weighted average of its
values:

$$\mathbb{E}[X] = \sum_{x} x\,P(X = x).$$

*In words:* each possible value counts in proportion to how probable it is. For a genotype $G \in \{0,1,2\}$ with
probabilities $(0.49, 0.42, 0.09)$: $\mathbb{E}[G] = 0 \times 0.49 + 1 \times 0.42 + 2 \times 0.09 = 0.6$ — the
average number of T alleles per person, which is $2q$ for allele frequency $q = 0.3$.

**Variance and standard deviation.** The variance is the expected squared distance from the mean, and the standard
deviation (sd) its square root (same units as $X$):

$$\operatorname{Var}(X) = \mathbb{E}\big[(X - \mathbb{E}[X])^2\big] = \mathbb{E}[X^2] - \mathbb{E}[X]^2, \qquad \operatorname{sd}(X) = \sqrt{\operatorname{Var}(X)}.$$

*In words:* how far values typically fall from the mean, squared (so that deviations above and below do not cancel).
For the genotype: $\mathbb{E}[G^2] = 1 \times 0.42 + 4 \times 0.09 = 0.78$, so $\operatorname{Var}(G) = 0.78 - 0.36 = 0.42 = 2q(1-q)$.

In [ ]:
# ── 5.1 Expectation and variance of a genotype: formula vs simulation ──
values, probs = np.array([0, 1, 2]), np.array([0.49, 0.42, 0.09])
E_G = np.sum(values * probs)
Var_G = np.sum(values ** 2 * probs) - E_G ** 2
print(f"formula:     E[G] = {E_G:.3f} (= 2q = {2 * 0.3:.3f}),  Var(G) = {Var_G:.3f} (= 2q(1−q) = {2 * 0.3 * 0.7:.3f})")
rng = np.random.default_rng(SEED + 51)
g = rng.choice(values, size=100_000, p=probs)
print(f"simulation:  mean = {g.mean():.3f},  variance = {g.var():.3f},  sd = {g.std():.3f}")

🔍 **What just happened?** The probability-weighted formulas give $\mathbb{E}[G] = 0.6$ and $\operatorname{Var}(G) = 0.42$, and the average and variance of 100,000 simulated genotypes agree to about two decimals. The *true* expectation
is a property of the distribution; the *sample mean* is a property of the data that estimates it.

### 5.2 Sample versions, covariance and correlation — with leukaemia genes

From data $x_1, \dots, x_N$ we compute the **sample mean** $\bar x = \frac1N\sum_i x_i$ and the **sample variance**
$s^2 = \frac{1}{N}\sum_i (x_i - \bar x)^2$. (Many textbooks divide by $N - 1$ instead, which removes a small bias;
NumPy's `np.var(x)` uses $N$, `np.var(x, ddof=1)` uses $N - 1$. For large $N$ the difference is negligible; Module 05
explains where it comes from.)

For two variables measured on the same samples, the **covariance** and the **correlation** describe how they vary
together:

$$\operatorname{cov}(x, y) = \frac{1}{N}\sum_{i=1}^{N}(x_i - \bar x)(y_i - \bar y), \qquad
r = \frac{\operatorname{cov}(x, y)}{s_x\, s_y}.$$

*In words:* for each sample, multiply "how far x is above its mean" by "how far y is above its mean"; if both tend to be
high together (and low together) the products are mostly positive and the covariance is positive. Dividing by the two
standard deviations gives the **correlation coefficient** $r$, a unit-free number between −1 and +1: +1 = perfect
increasing straight-line relationship, 0 = no linear relationship, −1 = perfect decreasing. We compute it for genes in
the leukaemia data of Modules 00–01.

> 🧪 **Simulated, with a known truth.** The course stores no data: the expression table is generated by the toolkit's
> `simulate_leukemia_expression`, shaped like the study of Golub *et al.* (1999). Because we know how the simulator
> works, we know which genes *truly* move together — something real data never tell us.

In [ ]:
# ── 5.2 Correlations between genes in the SIMULATED leukaemia data ──
leuk = simulate_leukemia_expression()
print("Source:", leuk["source"])
col = {gname: j for j, gname in enumerate(leuk["genes"])}
E = pd.DataFrame({gname: leuk["X"][:, col[gname]] for gname in ["ZYX", "CST3", "CD33", "CD79A", "G1551", "G2248"]})
cls = leuk["y"]                                                   # 0 = ALL, 1 = AML


def correlation(x: np.ndarray, y: np.ndarray) -> float:
    """Pearson correlation from the definition: covariance divided by the product of the standard deviations."""
    cov = np.mean((x - x.mean()) * (y - y.mean()))
    return cov / (x.std() * y.std())


for a, b in [("ZYX", "CST3"), ("ZYX", "CD33"), ("ZYX", "CD79A"), ("G1551", "G2248")]:
    within = [correlation(E.loc[cls == c, a].to_numpy(), E.loc[cls == c, b].to_numpy()) for c in (0, 1)]
    print(f"r({a}, {b}) = {correlation(E[a].to_numpy(), E[b].to_numpy()):+.3f}   (np.corrcoef: {np.corrcoef(E[a], E[b])[0, 1]:+.3f})"
          f"   | within ALL {within[0]:+.2f}, within AML {within[1]:+.2f}")
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
for ax, (a, b) in zip(axes, [("ZYX", "CST3"), ("ZYX", "CD79A"), ("G1551", "G2248")]):
    for c, name in enumerate(["ALL", "AML"]):
        m = cls == c
        ax.scatter(E.loc[m, a], E.loc[m, b], s=18, color=CLASS_COLORS[c], edgecolor="k", lw=0.3, label=name)
    ax.set_xlabel(f"{a} (log10 expression)"); ax.set_ylabel(f"{b} (log10 expression)")
    ax.set_title(f"r = {correlation(E[a].to_numpy(), E[b].to_numpy()):+.2f}  (72 SIMULATED patients)"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Our formula matches `np.corrcoef`. ZYX is **positively** correlated with CST3 and CD33 (all
three are higher in AML) and **negatively** with CD79A (higher in ALL). Look at the colours and at the within-class
numbers, though: inside the ALL group the three correlations are −0.04, −0.22 and −0.21 — small, within the range that
chance alone produces with 47 patients (roughly ±0.3) — and inside the AML group +0.47, +0.02 and −0.02. Most of each
correlation comes from the *difference between the two leukaemias*, not from the genes moving together within a
disease. A correlation computed on a mixture of groups can be driven by the groups rather than by any relationship
inside them. Keep that in mind for Section 5.4.

🧪 **What the simulator actually did.** Within a class, it draws each marker gene *independently* — the only shared
ingredient is the class difference. So why +0.47 for ZYX–CST3 within AML? The simulator makes two AML patients
**atypical**: their whole AML signature is weaker, so both genes are lower together in those two patients (the two red
points closest to the blue cloud in the first panel) — a hidden subgroup again. (Notice also the vertical stripe of
AML patients at ZYX = 4.2: the ceiling of the measurement scale.) The last pair is different: G1551 and G2248
are two unnamed genes that share one of the simulator's 15 hidden **co-expression programmes** and have no ALL/AML
difference at all (we found them by searching all pairs of such genes). Their correlation, +0.88, is just as strong
*within* each class (+0.89 and +0.86) as overall: these genes truly move together, whatever the leukaemia. Telling such direct co-expression apart from group-driven correlation is what
Gaussian graphical models do (Module 18).

### 5.3 Linearity of expectation and the variance of a sum

Two rules used constantly in the course (they follow from the definitions in a few lines):

$$\mathbb{E}[aX + bY] = a\,\mathbb{E}[X] + b\,\mathbb{E}[Y], \qquad
\operatorname{Var}(X + Y) = \operatorname{Var}(X) + \operatorname{Var}(Y) + 2\operatorname{cov}(X, Y).$$

*In words:* expectations simply add — **always**, whether or not X and Y are related. Variances add only if the
covariance is zero (e.g. for independent variables); if X and Y tend to move together, the sum varies *more*.

🧫 **Polygenic scores.** A polygenic score adds up risk alleles across SNPs. Two SNPs that sit close together on a
chromosome tend to be inherited together — they are in **linkage disequilibrium (LD)** — so their genotypes are
correlated. We simulate two SNPs (T-allele frequency 0.3 each), once unlinked and once in strong LD, and look at the
variance of their sum.

In [ ]:
# ── 5.3 Variance of a sum of two genotypes, without and with linkage disequilibrium ──
def two_snp_genotypes(rng, n: int, q: float, r_ld: float):
    """Genotypes at two SNPs from two haplotypes per person; with prob. r_ld the 2nd allele copies the 1st (LD)."""
    G1, G2 = np.zeros(n, int), np.zeros(n, int)
    for _ in range(2):                                        # two chromosomes per person
        a1 = rng.random(n) < q
        a2 = np.where(rng.random(n) < r_ld, a1, rng.random(n) < q)
        G1 += a1; G2 += a2
    return G1, G2


rng = np.random.default_rng(SEED + 53)
for r_ld in [0.0, 0.8]:
    G1, G2 = two_snp_genotypes(rng, 50_000, 0.3, r_ld)
    cov12 = np.mean((G1 - G1.mean()) * (G2 - G2.mean()))
    print(f"LD parameter {r_ld}: E[G1+G2] = {np.mean(G1 + G2):.3f} (E[G1]+E[G2] = {G1.mean() + G2.mean():.3f}) | "
          f"Var(G1+G2) = {np.var(G1 + G2):.3f} vs Var+Var+2cov = {G1.var() + G2.var() + 2 * cov12:.3f} "
          f"(cov = {cov12:.3f}, r = {np.corrcoef(G1, G2)[0, 1]:.2f})")

🔍 **What just happened?** In both cases the mean of the sum is the sum of the means (1.2 = 0.6 + 0.6): linearity of
expectation does not care about LD. The variance does: for unlinked SNPs the covariance is ≈ 0 and $\operatorname{Var} (G_1+G_2) \approx 0.42 + 0.42$; with strong LD ($r \approx 0.8$) the covariance adds about $2 \times 0.34$ and the variance
of the score nearly doubles (0.84 → 1.53). Correlated predictors are a recurring headache — they make regression coefficients
unstable (Modules 06–07).

### 5.4 Correlation is not causation: a population-structure confounder

🧫 Two populations differ in the frequency of a SNP's T allele (0.2 vs 0.7) and — for reasons that have nothing to do
with this SNP (diet, environment, other genes) — in the average value of a trait (0 vs 1 standard unit). The SNP has
**no effect** on the trait. We sample 1,000 people from each population and pool them, as a careless GWAS might.

In [ ]:
# ── 5.4 A SNP with no effect, correlated with a trait through population structure (SIMULATED) ──
rng = np.random.default_rng(SEED + 54)
n_per = 1000
pop_lab = np.repeat([0, 1], n_per)
g_snp = np.r_[rng.binomial(2, 0.2, n_per), rng.binomial(2, 0.7, n_per)]    # allele frequency differs by population
trait = np.r_[rng.normal(0.0, 1, n_per), rng.normal(1.0, 1, n_per)]         # trait mean differs by population only
print(f"pooled correlation r(genotype, trait) = {np.corrcoef(g_snp, trait)[0, 1]:+.3f}")
for k in (0, 1):
    m = pop_lab == k
    print(f"  within population {k}: r = {np.corrcoef(g_snp[m], trait[m])[0, 1]:+.3f}")
print("mean trait by genotype (pooled):", {gv: round(float(trait[g_snp == gv].mean()), 2) for gv in (0, 1, 2)})

fig, ax = plt.subplots(figsize=(6, 3.4))
jitter = rng.uniform(-0.15, 0.15, len(g_snp))
for k, colour in [(0, "tab:blue"), (1, "tab:orange")]:
    m = pop_lab == k
    ax.scatter(g_snp[m] + jitter[m], trait[m], s=4, alpha=0.35, color=colour, label=f"population {k}")
ax.plot([0, 1, 2], [trait[g_snp == gv].mean() for gv in (0, 1, 2)], "ko-", label="pooled mean per genotype")
ax.set_xticks([0, 1, 2]); ax.set_xticklabels(["AA", "AT", "TT"]); ax.set_xlabel("genotype at the SNP (jittered)")
ax.set_ylabel("trait (standard units)")
ax.set_title("a SNP with NO effect looks associated (SIMULATED)"); ax.legend(fontsize=7, loc="upper left")
plt.tight_layout(); plt.show()

🔍 **What just happened?** In the pooled sample the trait rises steadily with the number of T alleles (black line) and
the correlation is about **+0.30** — a "hit" a GWAS would report. Within each population the correlation is
essentially **zero**. The association is entirely produced by the **confounder** (population), which influences both
the genotype frequency and the trait — exactly the structure of smoking → yellow nails and lung cancer in Section 4.
This is **population stratification**, a real and famous source of false positives in genetic association studies.
The standard remedies — including the population's genetic background as a covariate, or principal components of the
genotypes (Module 14) — all amount to *conditioning on the confounder*.

⚠️ **Correlation ≠ causation**, in both directions: a correlation can arise without any causal effect (here), and a real
causal effect can be hidden by a confounder pushing the other way. Learned dependencies — a regression coefficient, a
network edge in Modules 17–18 — are statistical statements, not proofs of mechanism.

✏️ **Your turn.** Give the SNP a real effect: add `0.3 * g_snp` to `trait` and re-run. What are the within-population
correlations now? Then set both populations' trait means to 0: does the pooled correlation still overstate the effect?

---
## 6 · Hardy–Weinberg as a probability model

> 📍 **Where we are.** Module 01 used the genotype proportions $(1-p)^2$, $2p(1-p)$, $p^2$ as a recipe. With the sum
> and product rules we can now *derive* them. (Notation: textbooks call the frequency of allele A $p$ and of allele T
> $q = 1 - p$; with the course's coding AA = 0, AT = 1, TT = 2 the genotype counts T alleles.)

Assume random mating in a large population: each child receives one allele from the mother and one from the father,
each drawn **independently**, with $P(\text{A}) = p$ and $P(\text{T}) = q$. Then

$$P(\text{AA}) = p \cdot p = p^2, \qquad
P(\text{AT}) = \underbrace{p \cdot q}_{\text{A from mother, T from father}} + \underbrace{q \cdot p}_{\text{T from mother, A from father}} = 2pq, \qquad
P(\text{TT}) = q^2.$$

*In words:* the **product rule** for independent events gives each ordered pair of alleles; the **sum rule** adds the
two disjoint ways of being heterozygous. The three probabilities add up to $p^2 + 2pq + q^2 = (p + q)^2 = 1$. In the
language of Section 1, the genotype $G$ (number of T alleles) is $\text{Binomial}(2, q)$. With $q = 0.3$:
$(0.49, 0.42, 0.09)$ — the numbers we have used all along.

In [ ]:
# ── 6.1 Observed genotype frequencies vs Hardy–Weinberg predictions for 200 simulated SNPs ──
rng = np.random.default_rng(SEED + 61)
Gm, _, _ = simulate_genotypes(rng, n_samples=2000, n_snps=200)
q_hat = Gm.mean(axis=0) / 2                                              # estimated T-allele frequency per SNP
obs = np.stack([(Gm == k).mean(axis=0) for k in (0, 1, 2)])             # observed frequencies of AA, AT, TT
pred = np.stack([(1 - q_hat) ** 2, 2 * q_hat * (1 - q_hat), q_hat ** 2])
fig, ax = plt.subplots(figsize=(4.6, 4))
for k, (lab, col) in enumerate(zip(["AA: p²", "AT: 2pq", "TT: q²"], ["tab:blue", "tab:green", "tab:red"])):
    ax.scatter(pred[k], obs[k], s=10, color=col, label=lab)
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_xlabel("Hardy–Weinberg prediction from q̂"); ax.set_ylabel("observed genotype frequency")
ax.set_title("200 SNPs × 2,000 people (SIMULATED)"); ax.legend()
plt.tight_layout(); plt.show()
print(f"largest |observed − predicted| over all SNPs and genotypes: {np.abs(obs - pred).max():.3f}")

🔍 **What just happened?** For every one of 200 SNPs and all three genotypes, the observed frequency lies on the
diagonal: Hardy–Weinberg predicts genotype frequencies from a single number per SNP, the allele frequency. (In real
data, deviations flag genotyping errors, selection, inbreeding or — as Module 01's mini-project showed — hidden
population structure.)

### 6.2 Cystic-fibrosis carriers from the disease frequency

Cystic fibrosis affects roughly **1 in 2,500** newborns of Northern-European ancestry (published estimates range from
about 1 in 2,500 to 1 in 3,500). Treating all disease-causing *CFTR* variants as a single allele c with frequency $q$,
affected children are cc, so $q^2 \approx 1/2500$. What fraction of healthy people are carriers?

In [ ]:
# ── 6.2 From disease incidence to carrier frequency ──
for incidence in [1 / 2500, 1 / 3500]:
    q_c = np.sqrt(incidence)                       # q² = incidence  →  q = √incidence
    carriers = 2 * (1 - q_c) * q_c                 # 2pq
    print(f"incidence 1/{1 / incidence:.0f}: allele frequency q = {q_c:.4f}, carrier frequency 2pq = {carriers:.4f} "
          f"(≈ 1 in {1 / carriers:.0f}); carriers per affected person: {carriers / incidence:.0f}")

🔍 **What just happened?** An incidence of 1 in 2,500 implies an allele frequency of 1 in 50 and a carrier frequency of
about **1 in 26** (0.039) — the "roughly 1 in 25" used in Sections 1 and 3. There are about **98 healthy carriers for
every affected person**: almost all disease alleles of a recessive disease hide in healthy heterozygotes, which is why
carrier screening exists. ⚠️ This uses Hardy–Weinberg's assumptions (random mating, no selection, a large population)
and lumps hundreds of different *CFTR* variants into one allele, so it is an approximation — a good one, as genetic
counsellors' tables show.

✏️ **Your turn.** A recessive disease affects 1 in 10,000 births. What is the carrier frequency? If a known carrier has
a child with an unrelated partner from the population, what is the probability that the child is affected?

---
## 7 · The Gaussian in detail

> 📍 **Where we are.** Section 1.4 introduced the Gaussian density. It is the workhorse of the course: noise in linear
> regression (Module 06), classes in Gaussian naïve Bayes (Module 08), clusters in mixture models (Module 13), PCA
> (Module 14) and Gaussian graphical models (Module 18). Here are the facts you will use.

### 7.1 The 68–95–99.7 rule and z-scores

For any Gaussian, about **68 %** of values lie within $\mu \pm \sigma$, **95 %** within $\mu \pm 2\sigma$ and **99.7 %**
within $\mu \pm 3\sigma$. To compare a value with its distribution we compute its **z-score**:

$$z = \frac{x - \mu}{\sigma}.$$

*In words:* how many standard deviations $x$ lies above ($z > 0$) or below ($z < 0$) the mean. If $X$ is Gaussian, $Z$
follows the **standard normal** distribution $N(0, 1)$, so a single table (or `stats.norm`) answers every question.

In [ ]:
# ── 7.1 The 68–95–99.7 rule, exactly and by simulation; a z-score ──
rng = np.random.default_rng(SEED + 71)
z_sample = rng.normal(0, 1, 1_000_000)
for k in (1, 2, 3):
    exact = stats.norm.cdf(k) - stats.norm.cdf(-k)
    print(f"within ±{k} sd: exact {exact:.4f} | simulated {np.mean(np.abs(z_sample) < k):.4f}")
height = 185.0
z = (height - mu_h) / sd_h
print(f"\nheight {height} cm: z = ({height} − {mu_h}) / {sd_h} = {z:.2f};  P(taller) = {stats.norm.sf(z):.4f}")

🔍 **What just happened?** The exact fractions are 0.6827, 0.9545 and 0.9973, and a million simulated values agree.
A height of 185 cm in our illustrative N(170, 7²) population has $z \approx 2.14$: only about **1.6 %** of people are
taller. z-scores are also how we put genes on a common scale (the toolkit's `standardize`, Modules 00–01).

### 7.2 The central limit theorem: why polygenic traits are bell-shaped

Why are height, blood pressure or cholesterol approximately Gaussian? R. A. Fisher's answer (1918): such traits are
influenced by **many genes, each with a small effect**, plus the environment. The **central limit theorem (CLT)** says
that a sum of many independent random contributions is approximately Gaussian, *whatever* the distribution of each
contribution. A single genotype takes only the values 0, 1, 2 — nothing like a bell — but watch what happens when we add
up more and more SNPs.

In [ ]:
# ── 7.2 Sum of J genotypes (equal small effects) for J = 1, 2, 10, 100 ──
rng = np.random.default_rng(SEED + 72)
Gp, _, _ = simulate_genotypes(rng, n_samples=5000, n_snps=100, maf_range=(0.2, 0.2))   # every SNP: T frequency 0.2
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
grid = np.linspace(-4, 4, 200)
for ax, J in zip(axes, [1, 2, 10, 100]):
    score = Gp[:, :J].sum(axis=1).astype(float)              # genetic score = number of T alleles over J SNPs
    zscore = (score - score.mean()) / score.std()
    edges = (np.arange(score.min() - 0.5, score.max() + 1.5) - score.mean()) / score.std()   # one bar per score value
    ax.hist(zscore, bins=edges, density=True, color="lightgray", edgecolor="gray")
    ax.plot(grid, stats.norm.pdf(grid), color="tab:red", lw=1.5)
    ax.set_xlim(-4, 4); ax.set_xlabel("standardised genetic score")
    ax.set_title(f"J = {J} SNP{'s' if J > 1 else ''} (skewness {stats.skew(score):+.2f})")
axes[0].set_ylabel("density")
fig.suptitle("Central limit theorem: sums of genotypes become Gaussian (SIMULATED, red = N(0, 1))", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** To change only one thing, every SNP here has T-allele frequency 0.2. With one SNP the
"trait" takes three values and is clearly lopsided (most people carry no T allele); with two SNPs, five values; with ten,
a lumpy hill; with 100 SNPs the histogram is almost indistinguishable from the red standard-normal curve. The
**skewness** (a measure of asymmetry, 0 for a Gaussian, printed in the titles) shrinks from about +1 towards 0. This is how Mendelian inheritance of discrete alleles produces continuous, bell-shaped quantitative
traits — and why 02-620 models quantitative phenotypes with Gaussian noise (the regression lecture, Module 06). The toolkit's
`simulate_quantitative_trait` builds traits exactly this way (Module 06).

### 7.3 Two correlated genes: the 2-D Gaussian

Two genes in the same pathway are often **co-expressed**. A pair of Gaussian variables is described by two means, two
variances and one correlation $\rho$ ("rho"), collected in a **mean vector** and a **covariance matrix**:

$$\boldsymbol\mu = \begin{pmatrix}\mu_1\\ \mu_2\end{pmatrix}, \qquad
\Sigma = \begin{pmatrix}\sigma_1^2 & \rho\,\sigma_1\sigma_2\\ \rho\,\sigma_1\sigma_2 & \sigma_2^2\end{pmatrix}.$$

*In words:* the diagonal of $\Sigma$ holds each gene's variance, the off-diagonal entries their covariance. (Vectors and
matrices are Module 03's topic; for now read $\Sigma$ as a 2 × 2 table of numbers.) $\rho = 0$ gives a round cloud,
$\rho$ near ±1 a narrow tilted ellipse.

In [ ]:
# ── 7.3 Samples from 2-D Gaussians with different correlations ──
rng = np.random.default_rng(SEED + 73)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), sharex=True, sharey=True)
for ax, rho in zip(axes, [0.0, 0.6, 0.95]):
    Sigma = np.array([[1.0, rho], [rho, 1.0]])                 # both genes standardised: variance 1
    pts = rng.multivariate_normal(mean=[0, 0], cov=Sigma, size=600)
    ax.scatter(pts[:, 0], pts[:, 1], s=5, alpha=0.5, color="tab:purple")
    ax.set_title(f"ρ = {rho}: sample r = {np.corrcoef(pts.T)[0, 1]:.2f}")
    ax.set_xlabel("gene 1 (z-score)"); ax.set_aspect("equal")
axes[0].set_ylabel("gene 2 (z-score)")
plt.tight_layout(); plt.show()
print("covariance matrix for ρ = 0.6:\n", np.array([[1.0, 0.6], [0.6, 1.0]]))

🔍 **What just happened?** `rng.multivariate_normal` drew pairs of values with the requested covariance matrix; the
sample correlations (titles) are close to the chosen $\rho$. As $\rho$ grows the cloud stretches along the diagonal:
knowing gene 1 tells you more and more about gene 2. The directions of these ellipses are what PCA finds (Module 14);
mixtures of such ellipses are Module 13's clusters; and in Module 18, the *inverse* of $\Sigma$ reveals which genes are
directly connected in a network.

✏️ **Your turn.** (1) In cell 7.2 replace the sum by a sum of genotypes **multiplied by random effect sizes**
(`Gp[:, :J] @ rng.normal(0, 1, J)`): is the result still Gaussian for J = 100? (2) In cell 7.3 try `rho = -0.8`. What
does a negative correlation look like, and which gene pair in Section 5.2 behaves like that?

---
## 8 · Counting parameters: why structure is necessary

> 📍 **Where we are.** Section 2 described two or three variables with a full table. The naïve-Bayes and
> Bayesian-network lectures of 02-620 start from a sobering observation: full tables explode as variables are added.
> This section counts.

A full joint distribution over $n$ **binary** variables is a table with $2^n$ cells. Because the cells must sum to 1,
the last one is determined by the others, so the table has

$$2^n - 1 \text{ free parameters (binary variables)}, \qquad 3^n - 1 \text{ for } n \text{ genotypes with values } 0/1/2.$$

*In words:* every new variable doubles (or triples) the size of the table. The Bayesian-network lecture counts the
chain-rule factorisation of five binary variables of the alarm example: $16 + 8 + 4 + 2 + 1 = 31 = 2^5 - 1$ — the
chain rule alone saves nothing. The naïve-Bayes lecture: a full table $P(Y \mid X_1, \dots, X_{30})$ for 30 SNPs needs
$3^{30} \approx 2 \times 10^{14}$ numbers. **Conditional independence** is the way out: naïve Bayes needs $P(Y)$
(1 number) plus, for each SNP and each class, a 3-valued distribution (2 free numbers): $1 + 2 \times n \times 2$. The smoking network of Section 4 needs
$1 + 2 + 2 = 5$ numbers instead of $2^3 - 1 = 7$.

In [ ]:
# ── 8.1 Parameter counts: full tables vs naïve Bayes ──
ns = np.array([1, 2, 3, 5, 10, 20, 30])
counts_table = pd.DataFrame({"n variables": ns,
                             "binary joint 2^n − 1": 2.0 ** ns - 1,
                             "genotype joint 3^n − 1": 3.0 ** ns - 1,
                             "naïve Bayes 1 + 2·n·2": 1 + 4 * ns})
print(counts_table.to_string(index=False, formatters={c: "{:,.0f}".format for c in counts_table.columns[1:]}))
fig, ax = plt.subplots(figsize=(6, 3.2))
for col, colour in zip(counts_table.columns[1:], ["tab:blue", "tab:red", "tab:green"]):
    ax.plot(ns, counts_table[col], "o-", color=colour, label=col)
ax.set_yscale("log"); ax.set_xlabel("number of variables n"); ax.set_ylabel("free parameters (log scale)")
ax.set_title("full joint tables explode; conditional independence keeps models small"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** With 30 SNPs a full genotype table needs about $2 \times 10^{14}$ parameters — more than the
number of people who have ever lived (roughly $10^{11}$), by a factor of about two thousand — while naïve Bayes needs **121**. On the log scale the
full tables are straight lines (exponential growth), the naïve-Bayes count is almost flat (linear growth).

Counting is not just bookkeeping: every parameter must be **estimated from data**. The next cell shows what happens if
we try to fill a full genotype table with 1,000 people.

In [ ]:
# ── 8.2 How many genotype combinations does a sample of 1,000 people actually contain? ──
rng = np.random.default_rng(SEED + 82)
rows = []
for n_snp in [1, 2, 3, 5, 8, 10, 12]:
    Gs, _, _ = simulate_genotypes(rng, n_samples=1000, n_snps=n_snp)
    code_id = (Gs * 3 ** np.arange(n_snp)).sum(axis=1)               # one integer per genotype combination
    seen, per_cell = np.unique(code_id, return_counts=True)
    rows.append((n_snp, 3 ** n_snp, len(seen), len(seen) / 3 ** n_snp, int(np.median(per_cell))))
print(pd.DataFrame(rows, columns=["SNPs", "possible combinations", "combinations seen", "fraction seen",
                                  "median people per seen combination"]).to_string(index=False))

🔍 **What just happened?** With 1 to 3 SNPs every genotype combination occurs, many times. With 5 SNPs only about half
of the 243 combinations appear (132), typically with just 4 people each; with 12 SNPs, 1,000 people cover fewer than
0.2 % of the 531,441 combinations, each seen about once. A table estimated by counting would be **mostly empty**, and
the filled cells would rest on one or two people — useless for predicting anyone new. This is the statistical reason,
beyond the computational one, why every probabilistic model in 02-620 imposes **structure**: naïve Bayes (Module 08),
Bayesian networks (31 → 10 parameters for the alarm network, Module 15), hidden Markov models (Module 19),
sparse Gaussian graphical models (Module 18).

### 📘 Lecture notes — counting parameters — the chain rule versus domain knowledge

**Free parameters of a table.** A joint distribution over variables with $v_1, \dots, v_n$ values has
$\prod_i v_i - 1$ free parameters (the last cell is fixed by "sums to 1"): $2^n - 1$ for binary variables, $3^n - 1$
for genotypes. A conditional table $P(X \mid \text{parents})$ has $(v_X - 1) \times (\text{number of parent
configurations})$.

**The chain rule assumes nothing.** For five binary variables M, J, A, B, E (MaryCalls, JohnCalls, Alarm, Burglary,
Earthquake):

$$P(M, J, A, B, E) = P(M \mid J, A, B, E)\,P(J \mid A, B, E)\,P(A \mid B, E)\,P(B \mid E)\,P(E)$$

*In words:* each variable conditioned on all the ones after it — a fully connected directed graph. Parameters:
$16 + 8 + 4 + 2 + 1 = 31 = 2^5 - 1$, exactly as many as the full table: exponential complexity.

**Domain knowledge to the rescue.** If we know that burglaries and earthquakes cause the alarm and the alarm makes
John and Mary call (B → A ← E, A → J, A → M), the joint becomes $P(B)P(E)P(A \mid B, E)P(J \mid A)P(M \mid A)$ with
$1 + 1 + 4 + 2 + 2 = 10$ parameters (Module 15). Likewise naïve Bayes for $n$ SNPs: $2 \cdot n \cdot 2 + 1$
instead of $(3^n - 1) \cdot 2 + 1$; the smoking network: $1 + 2 + 2 = 5$ instead of $2^3 - 1 = 7$.

**Why it matters statistically.** Every parameter must be estimated from data; with 1,000 people a full table over
12 SNPs leaves more than 99.8 % of its cells empty (Section 8.2).

> 🎓 **Exam tip.** Be able to count free parameters of any table: (number of cells − 1) for a joint, and (values of the child −
    1) × (number of parent configurations) for each conditional table. The alarm network's 10 (Module 15) is the
    classic exam question.

## 📌 Key takeaways

- A **random variable** is described by a pmf (discrete: Bernoulli, categorical/multinoulli, binomial) or a pdf
  (continuous: Gaussian, where probability = area). Simulated frequencies approach the true probabilities as samples
  grow (**law of large numbers**).
- **Sum rule** $P(A) = \sum_b P(A, b)$; **product rule** $P(A, B) = P(A \mid B)P(B)$; **chain rule**
  $P(A, B, C) = P(A)P(B \mid A)P(C \mid A, B)$ — always true.
- **Bayes' rule**: posterior = likelihood × prior / evidence. Rare conditions make most positive tests
  false alarms (base-rate fallacy); posteriors become priors for the next observation.
- **Independence** $P(A, B) = P(A)P(B)$ and **conditional independence** $P(A, B \mid C) = P(A \mid C)P(B \mid C)$ are
  different: yellow nails and lung cancer are dependent, but independent given smoking. Naïve Bayes is the chain rule
  plus a conditional-independence assumption.
- **Expectation** is linear; **variances** add only without covariance; **correlation** measures linear association
  and can be created by a **confounder** (population stratification) or by mixing groups (leukaemia classes) —
  correlation is not causation.
- **Hardy–Weinberg** ($p^2, 2pq, q^2$) follows from independent allele draws; a recessive disease's alleles hide mostly
  in healthy carriers.
- The **Gaussian**: 68–95–99.7, z-scores, the **central limit theorem** (polygenic traits), and 2-D Gaussians with a
  covariance matrix.
- Full joint tables need $2^n - 1$ (or $3^n - 1$) parameters and cannot be estimated; **conditional independence**
  makes probabilistic models tractable.

---
## ✅ Checkpoint

1. A genotype has P(AA) = 0.36, P(AT) = 0.48, P(TT) = 0.16. What is the T-allele frequency q, E[G] and Var(G) (G = number of T alleles)?
   <details><summary>Answer</summary>q = 0.4 (√0.16; check 2·0.6·0.4 = 0.48). E[G] = 0.48 + 2·0.16 = 0.8 = 2q. E[G²] = 0.48 + 4·0.16 = 1.12, so Var(G) = 1.12 − 0.64 = 0.48 = 2q(1 − q).</details>
2. Using the census table of Section 2.3, compute P(W = rich) and P(W = rich | G = male, H > 40.5).
   <details><summary>Answer</summary>P(rich) = 0.0245895 + 0.0116293 + 0.0971295 + 0.105933 ≈ 0.2393. P(rich | male, > 40.5) = 0.105933 / (0.134106 + 0.105933) ≈ 0.441.</details>
3. A disease has prevalence 0.1 %. A test has sensitivity 0.98 and specificity 0.99. What is P(disease | +)?
   <details><summary>Answer</summary>P(+) = 0.98·0.001 + 0.01·0.999 = 0.00098 + 0.00999 = 0.01097. P(disease | +) = 0.00098 / 0.01097 ≈ 0.089 — under 9 %.</details>
4. In the smoking network of Section 4.2, why does P(L = 1 | Y = 1) differ from P(L = 1 | Y = 0) although Y has no arrow to L?
   <details><summary>Answer</summary>Because both depend on S. Observing yellow nails raises the probability that the person smokes, which raises the probability of lung cancer. Given S, Y carries no further information: Y ⊥ L | S, but Y and L are not marginally independent.</details>
5. Write P(X₁, X₂, X₃ | Y) using the chain rule, then simplify it under the naïve-Bayes assumption.
   <details><summary>Answer</summary>Chain rule: P(X₁ | X₂, X₃, Y) P(X₂ | X₃, Y) P(X₃ | Y). Naïve Bayes (Xᵢ ⊥ Xⱼ | Y): P(X₁ | Y) P(X₂ | Y) P(X₃ | Y).</details>
6. X and Y have variances 1 and 4 and correlation 0.5. What is Var(X + Y)? What if they were independent?
   <details><summary>Answer</summary>cov = 0.5·1·2 = 1, so Var(X + Y) = 1 + 4 + 2·1 = 7. Independent: 1 + 4 = 5.</details>
7. How many free parameters does a full joint distribution over 4 SNPs (values 0/1/2) and a binary disease status have? And a naïve-Bayes model of the same variables?
   <details><summary>Answer</summary>Full joint: 3⁴ · 2 − 1 = 161. Naïve Bayes: 1 (P(Y)) + 2 classes × 4 SNPs × 2 free numbers = 17.</details>
8. What does this code estimate?  `np.mean(L[Y == 1])`
   <details><summary>Answer</summary>The fraction of people with lung cancer among those with yellow nails, i.e. an estimate of P(L = 1 | Y = 1) (L and Y are 0/1 arrays, so the mean of L over the selected people is a proportion).</details>

## 🧠 Questions & discussion

1. A direct-to-consumer genetic test reports a variant that is '5 times more common in people with disease X'. Which probability is that — P(variant | disease) or P(disease | variant)? What else would you need to know to tell a customer their risk?
2. Naïve Bayes assumes that SNPs are independent given disease status. Name a biological reason this is false for SNPs close together on a chromosome, and guess what that does to the predicted probabilities (Module 08 discusses it).
3. In Section 5.4, conditioning on population removed a spurious association. Can conditioning on a variable ever *create* a spurious association? (Think of a study that only recruits hospital patients — Module 16's collider.)
4. The central limit theorem explains bell-shaped traits. Which traits would you expect *not* to be Gaussian, and why (think of Mendelian diseases, or genes with large effects)?
5. Frequentists treat probabilities as long-run frequencies; Bayesians as degrees of belief (MLE vs MAP, Module 05). Which interpretation did you use for the genetic counselling example of Section 3.4? Did it matter?

## 🚀 Bonus — going further

- **Exact versus simulated.** Replace the simulation of Section 3.2 by many smaller studies (1,000 people each) and plot the spread of the estimated P(disease | +). How many people does a study need for a ±0.02 precision?
- **Linkage disequilibrium as dependence.** Use `two_snp_genotypes` to build the joint table of two SNPs in LD and measure how far it is from the product of its marginals. Relate the size of the discrepancy to the LD parameter.
- **Expression data and the Gaussian.** Pick a gene from the simulated leukaemia data (e.g. `E["ZYX"]`), plot its log-expression histogram in AML patients with a fitted Gaussian (mean and sd of the data). Is it Gaussian? What does the ceiling at 4.2 do?
- **Probability and uncertainty in this repository.** [`../../FundamentalsLearningUncertainty/`](../../FundamentalsLearningUncertainty/) treats likelihood, priors and uncertainty in more depth; [`../../ConformalPrediction/`](../../ConformalPrediction/) shows a modern way to attach calibrated uncertainty to any prediction.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module 02 — Probability — date:
Input: simulated carriers, genotypes, cohorts, test results, families, smoking/nails/cancer network;
       SIMULATED leukaemia expression (shaped like Golub et al. 1999) for correlations
Output: pmfs/pdfs vs histograms; P(D|G) tables; P(disease|+); 2/3 carrier risk; dependence vs conditional independence; HWE checks; CLT histograms; parameter counts
Method: sum, product, chain and Bayes rules; independence checks P(a,b) = P(a)P(b); sample moments; simulation
Key equation: P(θ|D) = P(D|θ) P(θ) / P(D);  P(X1, X2 | Y) = P(X1|Y) P(X2|Y) under conditional independence
Key parameters: prevalence, sensitivity, specificity; CPTs of S, Y, L; allele frequency q; number of SNPs J
What changed when I modified the prevalence / P(Y|S) / the LD parameter:
One failure mode: (e.g., confusing P(D|G) with P(G|D); a confounder creating correlation)
Ground-truth check: which gene pairs truly co-express, which correlate only through the classes?
Connection to the 02-620 topics (which idea of the lecture this illustrates):
```

## 📚 References

- Carnegie Mellon University 02-620 *Machine Learning for Scientists* (course design S. Kim; instructors M. Xu and M. Zhang) — this module teaches the probability prerequisite and the probability reviewed in its introductory, naïve-Bayes and Bayesian-network lectures.
- Murphy K. P. (2012) *Machine Learning: A Probabilistic Perspective*. MIT Press — Chapter 2 (probability: rules, Bayes, independence, common distributions, the central limit theorem).
- Hastie T., Tibshirani R., Friedman J. (2009) *The Elements of Statistical Learning*, 2nd ed. Springer — Section 2.4 (statistical decision theory) for how P(Y|X) is used for prediction.
- Mitchell T. M. (1997) *Machine Learning*. McGraw-Hill — Chapter 6 (Bayesian learning, Bayes' theorem, naïve Bayes).
- Pearl J. (2009) *Causality: Models, Reasoning, and Inference*, 2nd ed. Cambridge University Press — confounding and conditional independence in graphs.
- Fisher R. A. (1918) The correlation between relatives on the supposition of Mendelian inheritance. *Transactions of the Royal Society of Edinburgh* 52:399–433.
- Hartl D. L., Clark A. G. (2007) *Principles of Population Genetics*, 4th ed. Sinauer — Hardy–Weinberg equilibrium, population structure.
- Price A. L., Patterson N. J., Plenge R. M., Weinblatt M. E., Shadick N. A., Reich D. (2006) Principal components analysis corrects for stratification in genome-wide association studies. *Nature Genetics* 38:904–909.
- Golub T. R. *et al.* (1999) Molecular classification of cancer: class discovery and class prediction by gene expression monitoring. *Science* 286:531–537.

## ➡️ Next

[Module 03 · Vectors and matrices: the shape of biological data](03_linear_algebra_for_data.ipynb) — probability
    describes uncertainty; linear algebra describes the *shape* of data tables: dot products, the matrix X of N samples
    × J features, covariance matrices, and why least squares breaks down when there are more SNPs than people.